# Direct Residual Baseline Checks

This notebook implements the first residual-channel concept-discovery baseline for Residual SCBM.

The baseline question is simple:

> Do the learned residual dimensions already behave like discovered hidden concepts?

Protocol:

1. Load residual outputs from the split folders: `train/`, `val/`, `test/`.
2. Load true synthetic hidden residual labels from the matching dataset split.
3. Assert row alignment for every split.
4. Choose residual-to-hidden matches on validation only.
5. Report final recovery on test.

This is analogous to the concept-bank matching step in `cem-concept-discovery`, but here the residual channel is treated as one global missing-concept space instead of splitting each parent concept.


In [801]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, SparsePCA


import os
import re

## Select Experiment

Set `EXPERIMENT_PATH` to a Residual SCBM run that contains split-wise residual files:

```text
EXPERIMENT_PATH/train/res_mu.pt
EXPERIMENT_PATH/val/res_mu.pt
EXPERIMENT_PATH/test/res_mu.pt
```

The default below points at the hard synthetic run from the current analysis.


In [802]:
# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/hard/"
#     "alpha_1.0_beta_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_200_epochs_2026-06-04_17-42-45_5c587"
# )

# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/easy/"
#     "alpha_1.0_beta_1.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_easy_hid20_R20_dense_200_epochs_2026-06-07_19-24-48_e1bd1"
# )

# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/easy/"
#     "alpha_1.0_beta_1.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_easy_hid20_R50_sparse_200_epochs_2026-06-08_17-54-06_422d0"
# )


# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/hard/"
#     "alpha_1.0_beta_1.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_hard_hid20_R50_sparse_200_epochs_2026-06-08_18-55-53_856ff"

# )

# EXPERIMENT_PATH = Path("experiments/scbm_residual/synthetic_res_scbm/hard/"
#     "alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_hard_hid20_R20_hidden_strong_200_epochs_2026-06-10_13-12-25_84c8e"
    
# )


# EXPERIMENT_PATH = Path("experiments/scbm_residual/synthetic_res_scbm/hard/"
#                        "alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_2026-06-10_13-13-13_9c899"
# )
       
# EXPERIMENT_PATH = Path("experiments/scbm_residual/synthetic_res_scbm/hard/" 
#                        "alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_save_c_res_2026-06-11_15-10-20_c7313"
# )            


EXPERIMENT_PATH = Path("experiments/scbm_residual/synthetic_res_scbm/hard/"
                       "alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_hard_hid20_R20_sparse_200_epochs_save_c_res_2026-06-15_22-56-15_16428"
)



CONCEPT_RESIDUAL_CHANNEL = True



assert EXPERIMENT_PATH.exists(), EXPERIMENT_PATH
EXPERIMENT_PATH


PosixPath('experiments/scbm_residual/synthetic_res_scbm/hard/alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_hard_hid20_R20_sparse_200_epochs_save_c_res_2026-06-15_22-56-15_16428')

## Helpers


In [ ]:
def read_config_from_log(experiment_path: Path):
    log_path = experiment_path / "log.txt"
    assert log_path.exists(), log_path
    with log_path.open("r") as f:
        first_line = f.readline().strip()
    return ast.literal_eval(first_line)


def get_dim_data_and_model(experiment_path: Path):
    cfg = read_config_from_log(experiment_path)
    num_concepts = cfg["data"].get("num_concepts")
    num_residuals = cfg["data"].get("num_residuals")
    hid_dim = cfg["data"]["hid_dim"]
    obs_dim = cfg["data"]["obs_dim"]
    if num_concepts is None or num_residuals is None:
        raise ValueError(f"Could not find num_concepts or num_residuals in config: {cfg}")
    return num_concepts, num_residuals, hid_dim, obs_dim




def get_data_dir_name_from_log(experiment_path: Path):
    log_path = experiment_path / "log.txt"
    assert log_path.exists(), log_path

    data_dir_line = None
    with log_path.open("r") as f:
        for line in f:
            if line.startswith("data_dir:"):
                data_dir_line = line.split("data_dir:", 1)[1].strip()
                break
            if line.startswith("Loading existing synthetic dataset from"):
                data_dir_line = line.split("Loading existing synthetic dataset from", 1)[1].strip()
                break

    if data_dir_line is None:
        raise ValueError(f"Could not find data directory in {log_path}")

    return Path(data_dir_line).name


def infer_data_path(experiment_path: Path):
    cfg = read_config_from_log(experiment_path)
    data_cfg = cfg["data"]
    dataset = data_cfg["dataset"]
    difficulty = data_cfg.get("experiment_type")
    data_dir_name = get_data_dir_name_from_log(experiment_path)



    candidate = Path("datasets") / dataset / difficulty / data_dir_name


    if candidate.exists():
        return candidate

    raise FileNotFoundError("Could not find data path. Tried: " + ", ".join(map(str, candidates)))


def load_tensor(path: Path):
    assert path.exists(), path
    return torch.load(path, map_location="cpu")


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def load_split(experiment_path: Path, data_path: Path, split: str, num_concepts=None, num_residuals=None):
    model_split = experiment_path / split
    data_split = data_path / split

    if not model_split.exists():
        raise FileNotFoundError(
            f"Missing {model_split}. Re-run inference/training after adding deterministic analysis loaders."
        )
    assert data_split.exists(), data_split
    


    c_res_mu = load_tensor(model_split / "c_res_mu.pt")
    concept_residual_probs = load_tensor(model_split / "concepts_residuals_pred_probs_mean.pt")
    concept_residual_probs_std = load_tensor(model_split / "concepts_residuals_pred_probs_std.pt")
    concept_residual_sample_mean = load_tensor(model_split / "concepts_residuals_sample_mean.pt")
    
    res_mu = c_res_mu[:, num_concepts:]
    c_mu = c_res_mu[:, :num_concepts]
    concept_probs = concept_residual_probs[:, :num_concepts]
    residual_probs = concept_residual_probs[:, num_concepts:]
    concept_probs_std = concept_residual_probs_std[:, :num_concepts]
    residual_probs_std = concept_residual_probs_std[:, num_concepts:]
    concept_sample_mean = concept_residual_sample_mean[:, :num_concepts]
    residual_sample_mean = concept_residual_sample_mean[:, num_concepts:]
        
        


    out = {
        "res_mu": res_mu,
        "c_mu": c_mu,
        "concept_probs": concept_probs,
        "residual_probs": residual_probs,
        "concept_probs_std": concept_probs_std,
        "residual_probs_std": residual_probs_std,
        "concept_sample_mean": concept_sample_mean,
        "residual_sample_mean": residual_sample_mean,
        "hidden_residuals": load_tensor(data_split / "residuals.pt"),
        "hidden_residual_signal": load_tensor(data_split / "residual_signal.pt"),
        "concepts": load_tensor(data_split / "concepts.pt"),
        "y": load_tensor(data_split / "y.pt"),
        "w_hid": load_tensor(data_split / "w_hid.pt"),
        "w_obs": load_tensor(data_split / "w_obs.pt"), 
    }



    
    n_model = out["res_mu"].shape[0]
    n_data = out["hidden_residuals"].shape[0]
    assert n_model == n_data, (
        f"{split} row mismatch: model residuals have {n_model} rows, "
        f"but hidden residual labels have {n_data}. The residual dump likely used a shuffled/drop_last loader."
    )
    assert out["residual_probs"].shape[0] == n_data
    assert out["residual_sample_mean"].shape[0] == n_data

    return out







def orientation_free_auc(y_true, score):
    y_true = to_numpy(y_true).astype(int)
    score = to_numpy(score).astype(float)
    if len(np.unique(y_true)) < 2:
        return np.nan, "degenerate"
    raw_auc = roc_auc_score(y_true, score)
    if raw_auc >= 0.5:
        return raw_auc, "positive"
    return 1.0 - raw_auc, "negative"


def residual_hidden_auc_matrix(residual_scores, hidden_residuals):
    R = to_numpy(residual_scores).astype(float)
    H = to_numpy(hidden_residuals).astype(int)

    auc = np.zeros((R.shape[1], H.shape[1]), dtype=float)
    directions = np.empty((R.shape[1], H.shape[1]), dtype=object)

    for r in range(R.shape[1]):
        for h in range(H.shape[1]):
            auc[r, h], directions[r, h] = orientation_free_auc(H[:, h], R[:, r])

    return auc, directions


def plot_auc_matrix(auc, title, relevant_hidden_indices=None):
    plt.figure(figsize=(1.15 * auc.shape[1] + 3, 0.95 * auc.shape[0] + 2))
    # Plot only relevenat hidden indices if provided, otherwise plot all
    auc = auc[:, relevant_hidden_indices] if relevant_hidden_indices is not None else auc
    
    
    sns.heatmap(
        auc,
        annot=True,
        fmt=".3f",
        vmin=0.5,
        vmax=1.0,
        cmap="viridis",
        xticklabels=[f"h{j}" for j in range(auc.shape[1])],
        yticklabels=[f"r{i}" for i in range(auc.shape[0])],
    )
    plt.xlabel("True hidden residual")
    plt.ylabel("Learned residual dimension")
    plt.title(title)
    plt.tight_layout()


def match_hidden_to_residuals(residual_scores, hidden_residuals):
    """
    For each hidden residual dimension, find the best matching residual dimension based on AUC.
    """
    auc, directions = residual_hidden_auc_matrix(residual_scores, hidden_residuals)
    rows = []

    # For each hidden residual dimension, find the residual dimension with the highest AUC 
    # and record the direction.
    for h in range(auc.shape[1]):
        r = int(np.nanargmax(auc[:, h]))
        rows.append({
            "hidden_idx": h,
            "residual_idx": r,
            "auc": float(auc[r, h]),
            "direction": directions[r, h],
        })

    return pd.DataFrame(rows), auc, directions


def evaluate_fixed_matches(residual_scores, hidden_residuals, matches, threshold=0.0):
    """If validation says residual dimension r_k corresponds to hidden concept h_j, 
    does that same residual dimension recover h_j on unseen test data"""
    
    
    R = to_numpy(residual_scores).astype(float)
    H = to_numpy(hidden_residuals).astype(int)
    rows = []
    
    # row = [hidden_idx, residual_idx, direction]
    for row in matches.itertuples(index=False):
        h = int(row.hidden_idx)
        r = int(row.residual_idx)
        # Score is for example res_mu if that is passed in as residual_scores
        score = R[:, r]
        direction = row.direction

        if direction == "negative":
            score = -score

        auc = roc_auc_score(H[:, h], score) if len(np.unique(H[:, h])) == 2 else np.nan
        # AUC does not depend on the threshold, but accuracy and F1 do
        pred = (score >= threshold).astype(int)

        rows.append({
            "hidden_idx": h,
            "residual_idx": r,
            "direction": direction,
            "auc": float(auc),
            "accuracy_at_threshold": float(accuracy_score(H[:, h], pred)),
            "f1_at_threshold": float(f1_score(H[:, h], pred)),
        })

    return pd.DataFrame(rows)


def add_task_relevance(df, w_hid):
    """Add a column to the hidden residuals dataframe indicating whether each hidden residual is task-relevant based on w_hid."""
    w_hid_np = to_numpy(w_hid).astype(float)
    df = df.copy()
    df["w_hid"] = w_hid_np[df["hidden_idx"].values]
    df["task_relevant"] = np.abs(df["w_hid"]) > 1e-8
    df["abs_w_hid"] = np.abs(df["w_hid"])
    df["rank_abs_w"] = df["abs_w_hid"].rank(method="min", ascending=False).astype(int)
    return df



def get_task_hidden_feature_name(experiment_path: Path):
    cfg = read_config_from_log(experiment_path)
    difficulty = cfg["data"].get("experiment_type")
    if difficulty == "hard":
        return "hidden_residual_signal"
    return "hidden_residuals"


def hidden_relevance_table(split_data, experiment_path: Path):
    """
    Estimate true hidden-concept task importance from the synthetic task weights.

    The primary importance metric is abs(w_hid): the absolute ground-truth
    coefficient assigned to each hidden concept by the data-generating process.
    Signal/prevalence statistics are retained only as descriptive diagnostics.
    """
    w = to_numpy(split_data["w_hid"]).astype(float)
    H = to_numpy(split_data["hidden_residuals"]).astype(float)
    S = to_numpy(split_data["hidden_residual_signal"]).astype(float)
    y = to_numpy(split_data["y"]).astype(int)
    task_feature_name = get_task_hidden_feature_name(experiment_path)
    task_feature = H if task_feature_name == "hidden_residuals" else S

    rows = []
    for h in range(H.shape[1]):
        # These empirical terms are useful diagnostics, but they do not define
        # task importance in this notebook. Importance is ranked by abs(w_hid).
        term = w[h] * task_feature[:, h]
        signal_term = w[h] * S[:, h]
        binary_term = w[h] * H[:, h]

        if len(np.unique(H[:, h])) == 2:
            hidden_auc_y, hidden_dir_y = orientation_free_auc(y, H[:, h])
        else:
            hidden_auc_y, hidden_dir_y = np.nan, "degenerate"

        rows.append({
            "hidden_idx": h,
            "w_hid": w[h],
            "abs_w_hid": abs(w[h]),
            "prevalence": H[:, h].mean(),
            "mean_signal": S[:, h].mean(),
            "std_binary_feature": H[:, h].std(),
            "std_signal_feature": S[:, h].std(),
            "task_feature_used_for_y": task_feature_name,
            "mean_abs_binary_term": np.abs(binary_term).mean(),
            "std_binary_term": binary_term.std(),
            "mean_abs_signal_term": np.abs(signal_term).mean(),
            "std_signal_term": signal_term.std(),
            "mean_abs_task_term": np.abs(term).mean(),
            "std_task_term": term.std(),
            "hidden_auc_vs_y": hidden_auc_y,
            "hidden_direction_vs_y": hidden_dir_y,
        })

    out = pd.DataFrame(rows)
    # Rank by absolute task weight, then by std of task term, then by hidden AUC vs y
    out["rank_abs_w"] = out["abs_w_hid"].rank(method="min", ascending=False).astype(int)
    out["rank_std_task_term"] = out["std_task_term"].rank(method="min", ascending=False).astype(int)
    out["rank_hidden_auc_vs_y"] = out["hidden_auc_vs_y"].rank(method="min", ascending=False).astype(int)
    return out.sort_values("rank_abs_w")

def attach_relevance(df, relevance):
    cols = [
        "hidden_idx",
        "w_hid",
        "abs_w_hid",
        "prevalence",
        "mean_signal",
        "std_task_term",
        "mean_abs_task_term",
        "hidden_auc_vs_y",
        "rank_abs_w",
        "rank_std_task_term",
        "rank_hidden_auc_vs_y",
    ]

    # Some upstream tables may already contain older relevance columns from
    # add_task_relevance(...). Drop them before merging so pandas does not create
    # confusing suffixes like w_hid_x / w_hid_y.
    df = df.copy()
    stale_cols = [col for col in cols if col != "hidden_idx" and col in df.columns]
    if stale_cols:
        df = df.drop(columns=stale_cols)

    return df.merge(relevance[cols], on="hidden_idx", how="left")

def summarize_recovery_by_relevance(df, score_col, relevance_col="abs_w_hid", ks=(1, 3, 5, 10)):
    """
    Summarize how well the top-k most relevant hidden concepts are recovered by a given score (e.g. residual-hidden AUC or probe coefficients).
    """
    rows = []
    ordered = df.sort_values(relevance_col, ascending=False)
    for k in ks:
        top = ordered.head(k)
        rows.append({
            "top_k_by": relevance_col,
            "k": k,
            f"mean_{score_col}": top[score_col].mean(),
            f"max_{score_col}": top[score_col].max(),
            f"num_{score_col}_ge_0_7": int((top[score_col] >= 0.7).sum()),
            "hidden_indices": top["hidden_idx"].tolist(),
        })
    return pd.DataFrame(rows)



# -------------------------------------------------------------
# Test metrics of concept model and linear model
# --------------------------------------------------------------
def get_metrics_dataset_linear_model(full_model_path, dataset_difficulty, dataset="synthetic_res_scbm", get_res_used_model=False):
    full_data_path = get_data_dir_name_from_log(full_model_path)
    data_dir_name = full_data_path.split("/")[-1]
    linear_models_dir = os.path.join("experiments", "linear_head", dataset, dataset_difficulty)
    if get_res_used_model:
        model_name_start = data_dir_name + "_trueResUsed_True"
    else:
        model_name_start = data_dir_name + "_trueResUsed_False"
    for model_dir in os.listdir(linear_models_dir):
        if model_dir.startswith(model_name_start):
            linear_model_path = model_dir
            break
    with open(os.path.join(linear_models_dir, linear_model_path, "log.txt"), "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Test" in line:
                test_line = line
                break
        y_accuracy = re.findall(r"Final Test Accuracy:\s*([0-9.]+)", test_line)[0]
        
    return float(y_accuracy)


def test_metrics(full_model_path):
    metrics = {}
    with open(os.path.join(full_model_path, "log.txt"), "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Test" in line:
                test_line = line
                break
        y_accuracy = re.findall(r"y_accuracy:\s*([0-9.]+)", test_line)[0]
        c_accuracy = re.findall(r"c_accuracy:\s*([0-9.]+)", test_line)[0]
        c_auc = re.findall(r"c_AUROC:\s*([0-9.]+)", test_line)[0]
        metrics["y_accuracy"] = float(y_accuracy)
        metrics["c_accuracy"] = float(c_accuracy)
        metrics["c_auc"] = float(c_auc)
    return metrics  










## Model performance

In [804]:
test_metrics_dict = test_metrics(EXPERIMENT_PATH)
test_metrics_dict["linear_model_y_accuracy_with_res"] = get_metrics_dataset_linear_model(EXPERIMENT_PATH, dataset_difficulty="hard", dataset="synthetic_res_scbm", get_res_used_model=True)
test_metrics_dict["linear_model_y_accuracy_no_res"] = get_metrics_dataset_linear_model(EXPERIMENT_PATH, dataset_difficulty="hard", dataset="synthetic_res_scbm", get_res_used_model=False)

df_test_metrics = pd.DataFrame([test_metrics_dict])
display(df_test_metrics)

,y_accuracy,c_accuracy,c_auc,linear_model_y_accuracy_with_res,linear_model_y_accuracy_no_res
0,0.798,0.739,0.812,0.7537,0.5525


## Load Split-Aligned Data

The shape assertions here are intentional. If they fail for train, the run was probably produced before the deterministic analysis loaders were added.


In [805]:
DATA_PATH = infer_data_path(EXPERIMENT_PATH)
print("Experiment:", EXPERIMENT_PATH)
print("Data path:", DATA_PATH)


num_concepts, num_residuals, _,_ = get_dim_data_and_model(EXPERIMENT_PATH)

if CONCEPT_RESIDUAL_CHANNEL:
    splits = {split: load_split(EXPERIMENT_PATH, DATA_PATH, split, num_concepts=num_concepts, num_residuals=num_residuals) 
              for split in ["train", "val", "test"]}

else:
    splits = {split: load_split(EXPERIMENT_PATH, DATA_PATH, split) for split in ["train", "val", "test"]}

for split, data in splits.items():
    print(f"{split}")
    for key in ["res_mu", "residual_probs", "residual_sample_mean", "hidden_residuals", "hidden_residual_signal", "concepts", "y"]:
        print(f"  {key:24s} {tuple(data[key].shape)}")


w_hid = splits["test"]["w_hid"]
w_obs = splits["test"]["w_obs"]
relevant_hidden = torch.where(w_hid.abs() > 1e-8)[0].tolist()


print("Task-relevant hidden residual indices:", relevant_hidden)



Experiment: experiments/scbm_residual/synthetic_res_scbm/hard/alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_hard_hid20_R20_sparse_200_epochs_save_c_res_2026-06-15_22-56-15_16428
Data path: datasets/synthetic_res_scbm/hard/cluster_a_0.5_b_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_r_sparsity_0.25_c_sparsity_0.3_sigmax_0.5_seed_0
train
  res_mu                   (30000, 20)
  residual_probs           (30000, 20)
  residual_sample_mean     (30000, 20)
  hidden_residuals         (30000, 20)
  hidden_residual_signal   (30000, 20)
  concepts                 (30000, 10)
  y                        (30000,)
val
  res_mu                   (10000, 20)
  residual_probs           (10000, 20)
  residual_sample_mean     (10000, 20)
  hidden_residuals         (10000, 20)
  hidden_residual_signal   (10000, 20)
  concepts                 (10000, 10)
  y                        (10000,)
test
  res_mu                   (10000, 20)
  residual_probs           (10000, 20)
  residual_sample_mean     (10000, 20)
  hi

In [ ]:
print("Concept and hidden concept abs weight comparison:")


print(f"w_hid.shape: {w_hid.shape}, w_obs.shape: {w_obs.shape}")

w_hid_and_w_obs = torch.cat([w_hid, w_obs], dim=0)



idx = [f"hid_{i}" for i in range(w_hid.shape[0])] + [f"obs_{i}" for i in range(w_obs.shape[0])]
df_w_hid_and_w_obs = pd.DataFrame({
    "idx": idx,
    "w_hid_and_w_obs": w_hid_and_w_obs.detach().cpu().numpy(),
    "abs_w_hid_and_w_obs": w_hid_and_w_obs.abs().detach().cpu().numpy(),
})

# only display non-zero abs weights
df_w_hid_and_w_obs = df_w_hid_and_w_obs[df_w_hid_and_w_obs["abs_w_hid_and_w_obs"] > 1e-8]
display(df_w_hid_and_w_obs.sort_values("abs_w_hid_and_w_obs", ascending=False))



Concept and hidden concept abs weight comparison:
w_hid.shape: torch.Size([20]), w_obs.shape: torch.Size([10])


,idx,w_hid_and_w_obs,abs_w_hid_and_w_obs
20,obs_0,0.916100,0.916100
18,hid_18,-0.705692,0.705692
2,hid_2,-0.648612,0.648612
23,obs_3,-0.384910,0.384910
9,hid_9,0.230210,0.230210
19,hid_19,-0.150507,0.150507
27,obs_7,0.112267,0.112267
7,hid_7,0.075182,0.075182


## Choose Matches On Validation, Evaluate On Test

This is the leakage-safe direct baseline. Validation chooses which learned residual dimension corresponds to each hidden residual. Test reports those fixed choices.

Use RES_MU


In [807]:
matches_train, auc_train, directions_train = match_hidden_to_residuals(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
)

matches_val, auc_val, directions_val = match_hidden_to_residuals(
    splits["val"]["res_mu"],
    splits["val"]["hidden_residuals"],
)

match_comparison = matches_val.merge(
    matches_train,
    on="hidden_idx",
    suffixes=("_val", "_train"),
)
match_comparison["same_residual_match"] = match_comparison["residual_idx_val"] == match_comparison["residual_idx_train"]
match_comparison = add_task_relevance(match_comparison.rename(columns={"hidden_idx": "hidden_idx"}), w_hid)


# Sort by abs(w_hid) to prioritize task-relevant hidden residuals
match_comparison = add_task_relevance(match_comparison, splits["test"]["w_hid"])
match_comparison = match_comparison.assign(abs_w_hid=match_comparison["w_hid"].abs())
match_comparison = match_comparison.sort_values("abs_w_hid", ascending=False)

display(match_comparison)



,hidden_idx,residual_idx_val,auc_val,direction_val,residual_idx_train,auc_train,direction_train,same_residual_match,w_hid,task_relevant,abs_w_hid,rank_abs_w
18,18,5,0.686339,positive,1,0.702285,negative,False,-0.705692,True,0.705692,1
2,2,2,0.715253,negative,16,0.730587,positive,False,-0.648612,True,0.648612,2
9,9,16,0.587137,negative,17,0.581822,positive,False,0.230210,True,0.230210,3
19,19,12,0.523878,negative,12,0.528995,negative,True,-0.150507,True,0.150507,4
7,7,5,0.529810,negative,18,0.530525,positive,False,0.075182,True,0.075182,5
12,12,17,0.533908,negative,17,0.530575,negative,True,0.000000,False,0.000000,6
17,17,4,0.528105,negative,4,0.530329,negative,True,0.000000,False,0.000000,6
16,16,2,0.515061,negative,2,0.511908,negative,True,0.000000,False,0.000000,6
15,15,6,0.527524,negative,3,0.524348,negative,False,0.000000,False,0.000000,6
14,14,12,0.539348,negative,2,0.540241,negative,False,0.000000,False,0.000000,6


In [808]:
# Evaluate the validation-chosen raw residual/hidden matches on the held-out test split.
test_eval = evaluate_fixed_matches(
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
    matches_val,
    threshold=0.0,
)

# Build one canonical hidden-concept relevance table for the rest of the notebook.
# Primary task relevance is abs(w_hid); the extra columns are descriptive diagnostics.
relevance = hidden_relevance_table(splits["test"], EXPERIMENT_PATH)

# Canonical raw-axis recovery table used by downstream comparison cells.
# Rename auc -> axis_auc so the column name is explicit when compared to probes/SAE.
recovery_eval = (
    attach_relevance(test_eval, relevance)
    .rename(columns={"auc": "axis_auc"})
    .sort_values("rank_abs_w")
)
recovery_eval["task_relevant"] = recovery_eval["abs_w_hid"] > 1e-8

print("Test evaluation of validation-chosen residual-hidden matches")
display(recovery_eval[[
    "hidden_idx",
    "residual_idx",
    "direction",
    "axis_auc",
    "accuracy_at_threshold",
    "f1_at_threshold",
    "w_hid",
    "abs_w_hid",
    "task_relevant",
]])

summary = pd.Series({
    "mean_auc_all_hidden": recovery_eval["axis_auc"].mean(),
    "mean_auc_task_relevant_hidden": recovery_eval.loc[recovery_eval["abs_w_hid"] > 1e-8, "axis_auc"].mean(),
    "num_hidden_auc_ge_0_7": int((recovery_eval["axis_auc"] >= 0.7).sum()),
    "num_task_relevant_hidden_auc_ge_0_7": int(((recovery_eval["axis_auc"] >= 0.7) & (recovery_eval["abs_w_hid"] > 1e-8)).sum()),
    "mean_accuracy_at_zero": recovery_eval["accuracy_at_threshold"].mean(),
})
summary


Test evaluation of validation-chosen residual-hidden matches


,hidden_idx,residual_idx,direction,axis_auc,accuracy_at_threshold,f1_at_threshold,w_hid,abs_w_hid,task_relevant
18,18,5,positive,0.687900,0.6447,0.638666,-0.705692,0.705692,True
2,2,2,negative,0.725691,0.6671,0.660618,-0.648612,0.648612,True
9,9,16,negative,0.586024,0.5602,0.572012,0.230210,0.230210,True
19,19,12,negative,0.527459,0.5219,0.511694,-0.150507,0.150507,True
7,7,5,negative,0.522412,0.5179,0.523570,0.075182,0.075182,True
4,4,4,positive,0.504254,0.5019,0.506098,0.000000,0.000000,False
5,5,6,negative,0.502981,0.4968,0.489862,0.000000,0.000000,False
6,6,18,negative,0.503160,0.4998,0.489383,0.000000,0.000000,False
8,8,7,negative,0.534018,0.5203,0.507647,0.000000,0.000000,False
1,1,10,positive,0.515386,0.5083,0.499134,0.000000,0.000000,False


mean_auc_all_hidden                    0.541894
mean_auc_task_relevant_hidden          0.609897
num_hidden_auc_ge_0_7                  1.000000
num_task_relevant_hidden_auc_ge_0_7    1.000000
mean_accuracy_at_zero                  0.529390
dtype: float64

## Raw axis

### Raw-Axis Pairwise AUC Table

This diagnostic computes every hidden-concept/residual-axis AUC pair, then displays only the two most task-relevant hidden concepts by `abs(w_hid)`.


In [809]:
# Compute every raw residual-axis / hidden-concept AUC pair on the test split.
# Each row is one candidate discovered residual concept (a raw residual axis)
# matched against one ground-truth hidden residual concept.
raw_axis_auc, raw_axis_directions = residual_hidden_auc_matrix(
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)

raw_axis_pair_rows = []
test_w_hid = splits["test"]["w_hid"]

for hidden_idx in range(raw_axis_auc.shape[1]):
    w_hid_value = float(test_w_hid[hidden_idx].item())
    for residual_idx in range(raw_axis_auc.shape[0]):
        raw_axis_pair_rows.append({
            "hidden_idx": hidden_idx,
            "residual_idx": residual_idx,
            "auc": float(raw_axis_auc[residual_idx, hidden_idx]),
            "direction": raw_axis_directions[residual_idx, hidden_idx],
            "w_hid": w_hid_value,
            "abs_w_hid": abs(w_hid_value),
        })

raw_axis_pair_auc_df = pd.DataFrame(raw_axis_pair_rows)
raw_axis_pair_auc_df["rank_abs_w"] = raw_axis_pair_auc_df["abs_w_hid"].rank(
    method="dense",
    ascending=False,
).astype(int)

# Sort first by hidden-concept task relevance, then by AUC within each hidden concept.
raw_axis_pair_auc_df = raw_axis_pair_auc_df.sort_values(
    ["rank_abs_w", "hidden_idx", "auc"],
    ascending=[True, True, False],
).reset_index(drop=True)

# Display only the two most task-relevant hidden concepts, but keep the full
# dataframe available as raw_axis_pair_auc_df for later analysis.
top2_task_hidden = (
    raw_axis_pair_auc_df[["hidden_idx", "abs_w_hid"]]
    .drop_duplicates("hidden_idx")
    .sort_values("abs_w_hid", ascending=False)
    .head(2)["hidden_idx"]
    .tolist()
)

print("Raw residual-axis AUC pairs for the two most task-relevant hidden concepts")
print("Top-2 hidden concepts by abs(w_hid):", top2_task_hidden)
display(raw_axis_pair_auc_df[raw_axis_pair_auc_df["hidden_idx"].isin(top2_task_hidden)])



Raw residual-axis AUC pairs for the two most task-relevant hidden concepts
Top-2 hidden concepts by abs(w_hid): [18, 2]


,hidden_idx,residual_idx,auc,direction,w_hid,abs_w_hid,rank_abs_w
0,18,1,0.689842,negative,-0.705692,0.705692,1
1,18,19,0.688735,positive,-0.705692,0.705692,1
2,18,13,0.688504,negative,-0.705692,0.705692,1
3,18,5,0.687900,positive,-0.705692,0.705692,1
4,18,11,0.687871,negative,-0.705692,0.705692,1
5,18,2,0.686691,negative,-0.705692,0.705692,1
6,18,14,0.686614,negative,-0.705692,0.705692,1
7,18,9,0.686307,positive,-0.705692,0.705692,1
8,18,7,0.686260,negative,-0.705692,0.705692,1
9,18,17,0.686181,negative,-0.705692,0.705692,1


## Distributed Probe Baseline

This asks whether hidden residual information is present in the residual vector even when it is not axis-aligned. The probe is trained on train residuals and evaluated on test residuals.


In [810]:
def distributed_probe_train_test(train_scores, train_hidden, test_scores, test_hidden):
    X_train = to_numpy(train_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)
    H_train = to_numpy(train_hidden).astype(int)
    H_test = to_numpy(test_hidden).astype(int)

    rows = []
    # For each hidden concept train a logistic regression on residual channel (res_mu)
    for h in range(H_train.shape[1]):
        if len(np.unique(H_train[:, h])) < 2 or len(np.unique(H_test[:, h])) < 2:
            continue

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
        )
        clf.fit(X_train, H_train[:, h])

        prob = clf.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        rows.append({
            "hidden_idx": h,
            "distributed_auc": roc_auc_score(H_test[:, h], prob),
            "distributed_accuracy": accuracy_score(H_test[:, h], pred),
            "distributed_f1": f1_score(H_test[:, h], pred),
        })

    return pd.DataFrame(rows)

probe_eval = distributed_probe_train_test(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)






probe_recovery_eval = add_task_relevance(probe_eval, splits["test"]["w_hid"])
#probe_recovery_eval = attach_relevance(probe_eval, relevance)





probe_recovery_eval.sort_values("rank_abs_w")


,hidden_idx,distributed_auc,distributed_accuracy,distributed_f1,w_hid,task_relevant,abs_w_hid,rank_abs_w
18,18,0.700613,0.6517,0.646432,-0.705692,True,0.705692,1
2,2,0.738841,0.6763,0.665703,-0.648612,True,0.648612,2
9,9,0.628799,0.5938,0.600276,0.230210,True,0.230210,3
19,19,0.575840,0.5526,0.557992,-0.150507,True,0.150507,4
7,7,0.571976,0.5487,0.542709,0.075182,True,0.075182,5
4,4,0.578416,0.5553,0.558260,0.000000,False,0.000000,6
5,5,0.596214,0.5706,0.566788,0.000000,False,0.000000,6
6,6,0.595253,0.5637,0.567635,0.000000,False,0.000000,6
8,8,0.579181,0.5594,0.565483,0.000000,False,0.000000,6
1,1,0.592108,0.5671,0.571429,0.000000,False,0.000000,6


## PCA Residual Concept Dictionary

This is the first UCBM-style dictionary-learning step adapted to the residual channel. PCA is fit only on `train/res_mu`, producing unsupervised residual concept directions. We then project train/val/test residual vectors into PCA concept scores. Hidden residual labels are used only after discovery, as a synthetic diagnostic of which hidden concepts the discovered directions align with.


In [811]:
# -----------------------------
# PCA residual concept dictionary
# -----------------------------
# Goal: learn an unsupervised dictionary C from residual vectors R = res_mu.
# PCA gives directions C that explain variance in train residual space. Each
# example is represented by concept scores Z = PCA.transform(R). This mirrors
# the UCBM idea of projecting model activations onto discovered concept vectors,
# but here the activations are residual-channel means.


def standardize_residual_numpy_from_train(train_scores, val_scores, test_scores):
    """Standardize residual vectors using train statistics only.

    This is part of the unsupervised dictionary fitting protocol: PCA sees only
    train residuals, and val/test are transformed with the same train mean/std.
    """
    X_train = to_numpy(train_scores).astype(np.float32)
    X_val = to_numpy(val_scores).astype(np.float32)
    X_test = to_numpy(test_scores).astype(np.float32)

    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std, mean, std


def fit_pca_residual_dictionary(X_train, X_val, X_test, n_components):
    """Fit a PCA dictionary on train residuals and project all splits.

    Returns a dictionary with the concept directions (`components`) and concept
    scores for each split. PCA components are unit-norm directions in residual
    space, so each score column can be treated as one discovered residual concept.
    """
    n_components = min(int(n_components), X_train.shape[1])
    pca = PCA(n_components=n_components, random_state=0)
    train_scores = pca.fit_transform(X_train)
    val_scores = pca.transform(X_val)
    test_scores = pca.transform(X_test)

    train_recon = pca.inverse_transform(train_scores)
    val_recon = pca.inverse_transform(val_scores)
    test_recon = pca.inverse_transform(test_scores)

    return {
        "method": "pca",
        "n_components": n_components,
        "model": pca,
        "components": pca.components_,
        "explained_variance_ratio_sum": float(pca.explained_variance_ratio_.sum()),
        "train_scores": train_scores,
        "val_scores": val_scores,
        "test_scores": test_scores,
        "train_reconstruction_mse": float(np.mean((X_train - train_recon) ** 2)),
        "val_reconstruction_mse": float(np.mean((X_val - val_recon) ** 2)),
        "test_reconstruction_mse": float(np.mean((X_test - test_recon) ** 2)),
    }


def pca_score_diagnostics(scores, eps=1e-6):
    """Unsupervised diagnostics for concept-score redundancy and sparsity."""
    scores = np.asarray(scores, dtype=float)
    if scores.shape[1] <= 1:
        mean_abs_corr = np.nan
    else:
        corr = np.corrcoef(scores, rowvar=False)
        off_diag = corr[~np.eye(corr.shape[0], dtype=bool)]
        mean_abs_corr = float(np.nanmean(np.abs(off_diag)))

    return {
        "mean_abs_score": float(np.mean(np.abs(scores))),
        "mean_active_rate_abs_gt_eps": float(np.mean(np.abs(scores) > eps)),
        "mean_abs_score_corr": mean_abs_corr,
    }


def match_dictionary_scores_on_val(val_scores, hidden_val):
    """Match each hidden residual to the best PCA concept using validation only.

    This is an evaluation step, not part of PCA discovery. It mirrors the raw-axis
    matching protocol: validation chooses a fixed discovered concept and direction;
    test evaluates that fixed choice.
    """
    auc, directions = residual_hidden_auc_matrix(val_scores, hidden_val)
    rows = []
    for h in range(auc.shape[1]):
        concept_idx = int(np.nanargmax(auc[:, h]))
        rows.append({
            "hidden_idx": h,
            "concept_idx": concept_idx,
            "val_auc": float(auc[concept_idx, h]),
            "direction": directions[concept_idx, h],
        })
    return pd.DataFrame(rows), auc, directions


def evaluate_fixed_dictionary_matches(test_scores, hidden_test, matches):
    """Evaluate validation-chosen PCA concept/hidden matches on test."""
    Z = np.asarray(test_scores, dtype=float)
    H = to_numpy(hidden_test).astype(int)
    rows = []
    for row in matches.itertuples(index=False):
        h = int(row.hidden_idx)
        c = int(row.concept_idx)
        score = Z[:, c]
        if row.direction == "negative":
            score = -score
        rows.append({
            "hidden_idx": h,
            "concept_idx": c,
            "direction": row.direction,
            "pca_auc": roc_auc_score(H[:, h], score),
            "val_auc": float(row.val_auc),
        })
    return pd.DataFrame(rows)


def pca_hidden_redundancy_table(test_scores, hidden_test, hidden_indices, thresholds=(0.6, 0.65, 0.7)):
    """Count how many PCA concepts align with each selected hidden concept.

    This helps distinguish a clean one-concept match from broad redundant signal.
    """
    auc, _ = residual_hidden_auc_matrix(test_scores, hidden_test)
    rows = []
    for h in hidden_indices:
        col = auc[:, h]
        top = np.argsort(-col)[: min(5, len(col))]
        row = {
            "hidden_idx": int(h),
            "best_pca_auc": float(np.nanmax(col)),
            "top5_pca_concepts": [int(c) for c in top],
            "top5_pca_aucs": [float(col[c]) for c in top],
        }
        for thr in thresholds:
            row[f"num_pca_concepts_auc_ge_{thr}"] = int(np.sum(col >= thr))
        rows.append(row)
    return pd.DataFrame(rows)


# Build standardized residual matrices once. No labels are used here.
X_train_pca_dict, X_val_pca_dict, X_test_pca_dict, pca_dict_mean, pca_dict_std = standardize_residual_numpy_from_train(
    splits["train"]["res_mu"],
    splits["val"]["res_mu"],
    splits["test"]["res_mu"],
)

# Start with a small sweep over dictionary sizes. k=20 matches the residual dimension
# in the active run and gives a rotation-style comparison to raw residual axes.
residual_dim = X_train_pca_dict.shape[1]
PCA_DICTIONARY_KS = sorted({k for k in [5, 10, 20, residual_dim] if 1 <= k <= residual_dim})

pca_dictionary_objects = {}
pca_dictionary_summaries = []
pca_dictionary_details = {}

for k in PCA_DICTIONARY_KS:
    pca_dict = fit_pca_residual_dictionary(X_train_pca_dict, X_val_pca_dict, X_test_pca_dict, k)
    pca_dictionary_objects[k] = pca_dict

    # Hidden labels are used only below this line, as a post-hoc synthetic diagnostic.
    matches_val, _, _ = match_dictionary_scores_on_val(
        pca_dict["val_scores"],
        splits["val"]["hidden_residuals"],
    )
    test_eval = evaluate_fixed_dictionary_matches(
        pca_dict["test_scores"],
        splits["test"]["hidden_residuals"],
        matches_val,
    )
    test_eval = attach_relevance(test_eval, relevance).sort_values("rank_abs_w")
    pca_dictionary_details[k] = test_eval

    score_diag = pca_score_diagnostics(pca_dict["test_scores"])
    pca_dictionary_summaries.append({
        "method": "pca",
        "n_components": k,
        "explained_variance_ratio_sum": pca_dict["explained_variance_ratio_sum"],
        "test_reconstruction_mse": pca_dict["test_reconstruction_mse"],
        **score_diag,
        "top1_mean_pca_auc": test_eval.head(1)["pca_auc"].mean(),
        "top3_mean_pca_auc": test_eval.head(3)["pca_auc"].mean(),
        "top5_mean_pca_auc": test_eval.head(5)["pca_auc"].mean(),
        "num_top5_hidden_auc_ge_0_7": int((test_eval.head(5)["pca_auc"] >= 0.7).sum()),
    })

pca_dictionary_summary = pd.DataFrame(pca_dictionary_summaries)
print("PCA dictionary unsupervised diagnostics plus post-hoc hidden recovery")
display(pca_dictionary_summary)

# Use the largest PCA dictionary as the default comparable dictionary for later summaries.
# This choice does not use hidden labels; it is simply the richest PCA dictionary in the sweep.
pca_dictionary_default_k = max(PCA_DICTIONARY_KS)
pca_dictionary_recovery_eval = pca_dictionary_details[pca_dictionary_default_k]

print(f"PCA dictionary recovery ranked by abs(w_hid), k={pca_dictionary_default_k}")
display(pca_dictionary_recovery_eval[[
    "hidden_idx",
    "concept_idx",
    "direction",
    "pca_auc",
    "val_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
]])

print("Raw residual axis vs PCA dictionary for task-relevant hidden concepts")
pca_vs_raw = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(pca_dictionary_recovery_eval[["hidden_idx", "pca_auc", "concept_idx"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)
display(pca_vs_raw.head(10))

relevant_hidden_ranked = pca_vs_raw.loc[pca_vs_raw["abs_w_hid"] > 1e-8, "hidden_idx"].tolist()
print("How many PCA concepts align with each task-relevant hidden concept?")
display(pca_hidden_redundancy_table(
    pca_dictionary_objects[pca_dictionary_default_k]["test_scores"],
    splits["test"]["hidden_residuals"],
    relevant_hidden_ranked,
))



PCA dictionary unsupervised diagnostics plus post-hoc hidden recovery


,method,n_components,explained_variance_ratio_sum,test_reconstruction_mse,mean_abs_score,mean_active_rate_abs_gt_eps,mean_abs_score_corr,top1_mean_pca_auc,top3_mean_pca_auc,top5_mean_pca_auc,num_top5_hidden_auc_ge_0_7
0,pca,5,0.986464,1.359828e-02,0.858012,1.000000,0.006792,0.687985,0.664151,0.607966,1
1,pca,10,0.993041,6.943718e-03,0.493318,1.000000,0.010531,0.687985,0.664151,0.607966,1
2,pca,20,1.000000,2.071903e-14,0.292977,0.999995,0.009673,0.687985,0.664151,0.611444,1


PCA dictionary recovery ranked by abs(w_hid), k=20


,hidden_idx,concept_idx,direction,pca_auc,val_auc,w_hid,abs_w_hid,rank_abs_w
18,18,0,negative,0.687985,0.684010,-0.705692,0.705692,1
2,2,0,negative,0.724859,0.713911,-0.648612,0.648612,2
9,9,0,positive,0.579610,0.581469,0.230210,0.230210,3
19,19,19,positive,0.532483,0.533719,-0.150507,0.150507,4
7,7,19,positive,0.532285,0.531014,0.075182,0.075182,5
4,4,17,negative,0.523381,0.530224,0.000000,0.000000,6
5,5,17,positive,0.540816,0.549856,0.000000,0.000000,6
6,6,12,positive,0.545982,0.551970,0.000000,0.000000,6
8,8,9,negative,0.541264,0.542722,0.000000,0.000000,6
1,1,15,negative,0.528874,0.543859,0.000000,0.000000,6


Raw residual axis vs PCA dictionary for task-relevant hidden concepts


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,pca_auc,concept_idx
0,18,0.687900,0.705692,1,0.687985,0
1,2,0.725691,0.648612,2,0.724859,0
2,9,0.586024,0.230210,3,0.579610,0
3,19,0.527459,0.150507,4,0.532483,19
4,7,0.522412,0.075182,5,0.532285,19
17,17,0.532613,0.000000,6,0.524527,16
16,16,0.515068,0.000000,6,0.541894,11
15,15,0.516571,0.000000,6,0.541992,17
14,14,0.533480,0.000000,6,0.531162,0
13,13,0.530722,0.000000,6,0.546217,7


How many PCA concepts align with each task-relevant hidden concept?


,hidden_idx,best_pca_auc,top5_pca_concepts,top5_pca_aucs,num_pca_concepts_auc_ge_0.6,num_pca_concepts_auc_ge_0.65,num_pca_concepts_auc_ge_0.7
0,18,0.687985,"[0, 4, 8, 16, 1]","[0.6879845812704042, 0.5361825483598923, 0.535...",1,1,0
1,2,0.724859,"[0, 6, 3, 8, 16]","[0.7248594476666086, 0.5363023494651992, 0.534...",1,1,1
2,9,0.579610,"[0, 8, 12, 17, 19]","[0.5796099650960865, 0.5611304928360854, 0.534...",0,0,0
3,19,0.536191,"[16, 19, 3, 10, 0]","[0.536191299129913, 0.5324829682968297, 0.5263...",0,0,0
4,7,0.532633,"[12, 19, 10, 0, 6]","[0.5326334408280342, 0.5322851799347281, 0.523...",0,0,0


## Task-Guided Residual Direction Discovery

This section turns the recoverability diagnostic into a discovery protocol. Candidate residual directions are learned without hidden labels, then a sparse task head selects directions using only `y`. Hidden labels are used only afterward to evaluate whether the selected directions recover the most task-relevant hidden concepts.


In [812]:
# -----------------------------
# Task-guided residual direction discovery
# -----------------------------
# Goal: discover task-relevant residual directions without using hidden labels.
#
# The protocol is intentionally split into three stages:
# 1. Candidate generation from res_mu only: raw axes, PCA, SparsePCA, and a combined pool.
# 2. Task selection using only y: train a sparse logistic head from candidate scores to y.
# 3. Hidden-concept evaluation: after discovery, match selected directions to hidden concepts
#    on validation and report fixed-match recovery on test.
#
# This makes the method a genuine task-guided discovery method: hidden labels are not used
# to create, rank, or select candidate directions.

TASK_DISCOVERY_N_COMPONENTS = min(50, to_numpy(splits["train"]["res_mu"]).shape[1])
TASK_DISCOVERY_TOP_MS = [5, 10, 20, 50]
TASK_HEAD_C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
TASK_HEAD_AUC_TOL = 0.005
TASK_DIRECTION_NORM_EPS = 1e-8


def standardize_residual_splits_for_discovery(train_scores, val_scores, test_scores):
    """Standardize residual vectors using train statistics only."""
    X_train = to_numpy(train_scores).astype(float)
    X_val = to_numpy(val_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)

    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std, mean, std


def normalize_direction_rows(directions):
    """Normalize concept/direction vectors so scores are comparable across generators."""
    directions = np.asarray(directions, dtype=float)
    norms = np.linalg.norm(directions, axis=1, keepdims=True) + TASK_DIRECTION_NORM_EPS
    return directions / norms


def build_residual_direction_candidates(X_train_std, n_components=TASK_DISCOVERY_N_COMPONENTS):
    """Create candidate residual directions without using y or hidden labels."""
    n_dims = X_train_std.shape[1]
    n_components = min(n_components, n_dims)

    # Raw axes: each original residual coordinate is a candidate concept direction.
    raw_dirs = np.eye(n_dims)

    # PCA directions: orthogonal directions that explain variance in the residual channel.
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_train_std)
    pca_dirs = pca.components_

    # SparsePCA directions: variance-seeking directions with sparse loadings.
    # These are a direct test of whether residual concepts are sparse rotated axes.
    sparse_pca = SparsePCA(
        n_components=n_components,
        alpha=1.0,
        ridge_alpha=0.01,
        max_iter=1000,
        tol=1e-4,
        random_state=0,
        n_jobs=1,
    )
    sparse_pca.fit(X_train_std)
    sparse_pca_dirs = sparse_pca.components_

    candidate_sets = {
        "raw_axes": normalize_direction_rows(raw_dirs),
        "pca": normalize_direction_rows(pca_dirs),
        "sparse_pca": normalize_direction_rows(sparse_pca_dirs),
    }

    # Combined pool lets the task head choose among all simple candidate generators.
    candidate_sets["combined_raw_pca_sparsepca"] = normalize_direction_rows(
        np.vstack([candidate_sets["raw_axes"], candidate_sets["pca"], candidate_sets["sparse_pca"]])
    )
    return candidate_sets


def project_residual_directions(X_std, directions):
    """Convert residual vectors into discovered concept scores."""
    return X_std @ directions.T


def fit_sparse_task_head(Z_train, y_train, Z_val, y_val, c_grid=TASK_HEAD_C_GRID):
    """
    Train sparse task heads and choose a model using validation y-AUC.

    Selection uses only downstream labels y. If several C values are effectively tied,
    choose the one with fewer nonzero concept weights for interpretability.
    """
    y_train = to_numpy(y_train).astype(int).reshape(-1)
    y_val = to_numpy(y_val).astype(int).reshape(-1)
    rows = []

    for C in c_grid:
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,
                C=C,
                class_weight="balanced",
                max_iter=5000,
                tol=1e-3,
                random_state=0,
            ),
        )
        clf.fit(Z_train, y_train)
        val_prob = clf.predict_proba(Z_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)
        coef = clf.named_steps["logisticregression"].coef_[0]
        n_nonzero = int((np.abs(coef) > 1e-8).sum())
        rows.append({"C": C, "val_task_auc": val_auc, "n_nonzero": n_nonzero, "clf": clf, "coef": coef})

    best_auc = max(row["val_task_auc"] for row in rows)
    eligible = [row for row in rows if row["val_task_auc"] >= best_auc - TASK_HEAD_AUC_TOL]
    best = min(eligible, key=lambda row: (row["n_nonzero"], -row["val_task_auc"], row["C"]))

    coef = best["coef"]
    ranked_direction_ids = np.argsort(-np.abs(coef)).astype(int).tolist()
    return best, pd.DataFrame([{k: v for k, v in row.items() if k not in ["clf", "coef"]} for row in rows]), ranked_direction_ids


def match_selected_directions_on_val(Z_val, hidden_val, selected_direction_ids):
    """For each hidden concept, choose the selected discovered direction with best val AUC."""
    H_val = to_numpy(hidden_val).astype(int)
    selected_scores = Z_val[:, selected_direction_ids]
    auc, directions = residual_hidden_auc_matrix(selected_scores, H_val)
    rows = []

    for h in range(H_val.shape[1]):
        local_idx = int(np.nanargmax(auc[:, h]))
        discovered_idx = int(selected_direction_ids[local_idx])
        rows.append({
            "hidden_idx": h,
            "selected_local_idx": local_idx,
            "discovered_direction_idx": discovered_idx,
            "val_discovery_auc": float(auc[local_idx, h]),
            "direction": directions[local_idx, h],
        })
    return pd.DataFrame(rows)


def evaluate_fixed_direction_matches_on_test(Z_test, hidden_test, matches):
    """Evaluate validation-chosen discovered-direction/hidden matches on held-out test."""
    H_test = to_numpy(hidden_test).astype(int)
    rows = []

    for row in matches.itertuples(index=False):
        h = int(row.hidden_idx)
        j = int(row.discovered_direction_idx)
        score = Z_test[:, j]
        if row.direction == "negative":
            score = -score

        # AUC evaluates ranking quality. Accuracy/F1 use a simple zero threshold
        # because candidate scores are centered by train standardization and projections.
        pred = (score >= 0.0).astype(int)
        rows.append({
            "hidden_idx": h,
            "discovered_direction_idx": j,
            "direction": row.direction,
            "val_discovery_auc": row.val_discovery_auc,
            "discovery_auc": roc_auc_score(H_test[:, h], score),
            "discovery_accuracy_at_zero": accuracy_score(H_test[:, h], pred),
            "discovery_f1_at_zero": f1_score(H_test[:, h], pred, zero_division=0),
        })
    return pd.DataFrame(rows)


# Stage 1: create candidate directions from train residuals only.
X_train_disc, X_val_disc, X_test_disc, disc_mean, disc_std = standardize_residual_splits_for_discovery(
    splits["train"]["res_mu"],
    splits["val"]["res_mu"],
    splits["test"]["res_mu"],
)
candidate_direction_sets = build_residual_direction_candidates(X_train_disc)

# Stage 2/3: for each candidate set, select directions using y, then evaluate hidden recovery.
task_guided_details = []
task_guided_summaries = []
task_head_tuning_tables = {}

for generator_name, directions in candidate_direction_sets.items():
    print("=" * 80)
    print(f"Candidate generator: {generator_name} | directions: {directions.shape[0]}")

    Z_train = project_residual_directions(X_train_disc, directions)
    Z_val = project_residual_directions(X_val_disc, directions)
    Z_test = project_residual_directions(X_test_disc, directions)

    task_head, tuning_table, ranked_direction_ids = fit_sparse_task_head(
        Z_train,
        splits["train"]["y"],
        Z_val,
        splits["val"]["y"],
    )
    task_head_tuning_tables[generator_name] = tuning_table

    test_task_prob = task_head["clf"].predict_proba(Z_test)[:, 1]
    test_task_auc = roc_auc_score(to_numpy(splits["test"]["y"]).astype(int), test_task_prob)
    print(
        f"selected C={task_head['C']} | val y-AUC={task_head['val_task_auc']:.3f} | "
        f"test y-AUC={test_task_auc:.3f} | nonzero directions={task_head['n_nonzero']}"
    )

    # Evaluate each effective top_m once. If there are only 10 candidate
    # directions, requested values 10/20/50 would otherwise all become 10 and
    # create duplicate rows that corrupt top-k summaries.
    effective_top_ms = sorted({min(m, len(ranked_direction_ids)) for m in TASK_DISCOVERY_TOP_MS})
    for top_m in effective_top_ms:
        selected_direction_ids = ranked_direction_ids[:top_m]

        # Hidden labels enter only here, after candidate generation and task selection.
        val_matches = match_selected_directions_on_val(
            Z_val,
            splits["val"]["hidden_residuals"],
            selected_direction_ids,
        )
        test_eval = evaluate_fixed_direction_matches_on_test(
            Z_test,
            splits["test"]["hidden_residuals"],
            val_matches,
        )
        test_eval = attach_relevance(test_eval, relevance).sort_values("rank_abs_w")
        test_eval["generator"] = generator_name
        test_eval["top_m_task_selected_directions"] = top_m
        test_eval["task_head_val_auc"] = task_head["val_task_auc"]
        test_eval["task_head_test_auc"] = test_task_auc
        test_eval["task_head_nonzero_directions"] = task_head["n_nonzero"]
        test_eval["selected_direction_ids"] = [selected_direction_ids] * len(test_eval)
        task_guided_details.append(test_eval)

        top1 = test_eval.head(1)
        top3 = test_eval.head(3)
        top5 = test_eval.head(5)
        task_guided_summaries.append({
            "generator": generator_name,
            "top_m_task_selected_directions": top_m,
            "task_head_val_auc": task_head["val_task_auc"],
            "task_head_test_auc": test_task_auc,
            "task_head_nonzero_directions": task_head["n_nonzero"],
            "top1_mean_discovery_auc": top1["discovery_auc"].mean(),
            "top3_mean_discovery_auc": top3["discovery_auc"].mean(),
            "top5_mean_discovery_auc": top5["discovery_auc"].mean(),
            "top5_unique_matched_directions": int(test_eval.head(5)["discovered_direction_idx"].nunique()),
            "top5_matched_direction_ids": test_eval.head(5)["discovered_direction_idx"].tolist(),
        })

task_guided_recovery_eval = pd.concat(task_guided_details, ignore_index=True)
task_guided_summary = pd.DataFrame(task_guided_summaries).sort_values(
    ["top5_mean_discovery_auc", "top3_mean_discovery_auc", "top5_unique_matched_directions"],
    ascending=[False, False, False],
)

print("Task-guided residual direction discovery summary")
display(task_guided_summary)

best_task_guided = task_guided_summary.iloc[0]
best_task_guided_detail = task_guided_recovery_eval[
    (task_guided_recovery_eval["generator"] == best_task_guided["generator"])
    & (task_guided_recovery_eval["top_m_task_selected_directions"] == best_task_guided["top_m_task_selected_directions"])
].sort_values("rank_abs_w")

print("Best task-guided discovery detail ranked by abs(w_hid)")
display(best_task_guided_detail[[
    "generator",
    "top_m_task_selected_directions",
    "hidden_idx",
    "discovered_direction_idx",
    "discovery_auc",
    "val_discovery_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
    "task_head_test_auc",
]])

# Compare the best task-guided discovery result to existing baselines.
task_guided_comparison = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(best_task_guided_detail[["hidden_idx", "discovery_auc", "discovered_direction_idx"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

print("Raw axis vs dense probe vs best task-guided discovery")
display(task_guided_comparison)

best_task_guided_summary = pd.DataFrame({
    "method": ["raw_axis", "dense_probe", "task_guided_discovery"],
    "top1_mean_auc": [
        task_guided_comparison.head(1)["axis_auc"].mean(),
        task_guided_comparison.head(1)["distributed_auc"].mean(),
        task_guided_comparison.head(1)["discovery_auc"].mean(),
    ],
    "top3_mean_auc": [
        task_guided_comparison.head(3)["axis_auc"].mean(),
        task_guided_comparison.head(3)["distributed_auc"].mean(),
        task_guided_comparison.head(3)["discovery_auc"].mean(),
    ],
    "top5_mean_auc": [
        task_guided_comparison.head(5)["axis_auc"].mean(),
        task_guided_comparison.head(5)["distributed_auc"].mean(),
        task_guided_comparison.head(5)["discovery_auc"].mean(),
    ],
})
display(best_task_guided_summary)


Candidate generator: raw_axes | directions: 20
selected C=0.001 | val y-AUC=0.875 | test y-AUC=0.882 | nonzero directions=15
Candidate generator: pca | directions: 20
selected C=0.001 | val y-AUC=0.875 | test y-AUC=0.882 | nonzero directions=1
Candidate generator: sparse_pca | directions: 20
selected C=0.001 | val y-AUC=0.876 | test y-AUC=0.882 | nonzero directions=11
Candidate generator: combined_raw_pca_sparsepca | directions: 60
selected C=0.001 | val y-AUC=0.875 | test y-AUC=0.882 | nonzero directions=8
Task-guided residual direction discovery summary


,generator,top_m_task_selected_directions,task_head_val_auc,task_head_test_auc,task_head_nonzero_directions,top1_mean_discovery_auc,top3_mean_discovery_auc,top5_mean_discovery_auc,top5_unique_matched_directions,top5_matched_direction_ids
12,combined_raw_pca_sparsepca,50,0.875443,0.882172,8,0.687900,0.666538,0.612883,4,"[42, 51, 16, 39, 39]"
4,pca,10,0.875420,0.882043,1,0.687985,0.664151,0.612252,3,"[0, 0, 0, 16, 12]"
5,pca,20,0.875420,0.882043,1,0.687985,0.664151,0.611451,2,"[0, 0, 0, 19, 19]"
11,combined_raw_pca_sparsepca,20,0.875443,0.882172,8,0.687900,0.664052,0.611392,3,"[42, 45, 42, 39, 39]"
3,pca,5,0.875420,0.882043,1,0.687985,0.664151,0.609939,2,"[0, 0, 0, 16, 0]"
2,raw_axes,20,0.875377,0.882153,15,0.687900,0.666538,0.609897,4,"[5, 2, 16, 12, 5]"
1,raw_axes,10,0.875377,0.882153,15,0.689842,0.666869,0.609414,5,"[1, 15, 16, 9, 19]"
7,sparse_pca,10,0.875558,0.882142,11,0.688735,0.666817,0.609383,4,"[7, 11, 19, 15, 7]"
8,sparse_pca,20,0.875558,0.882142,11,0.687900,0.666538,0.609233,4,"[2, 11, 19, 15, 2]"
0,raw_axes,5,0.875377,0.882153,15,0.688735,0.666482,0.609182,3,"[19, 16, 16, 9, 19]"


Best task-guided discovery detail ranked by abs(w_hid)


,generator,top_m_task_selected_directions,hidden_idx,discovered_direction_idx,discovery_auc,val_discovery_auc,w_hid,abs_w_hid,rank_abs_w,task_head_test_auc
240,combined_raw_pca_sparsepca,50,18,42,0.687900,0.686339,-0.705692,0.705692,1,0.882172
241,combined_raw_pca_sparsepca,50,2,51,0.725691,0.715253,-0.648612,0.648612,2,0.882172
242,combined_raw_pca_sparsepca,50,9,16,0.586024,0.587137,0.230210,0.230210,3,0.882172
243,combined_raw_pca_sparsepca,50,19,39,0.532505,0.533772,-0.150507,0.150507,4,0.882172
244,combined_raw_pca_sparsepca,50,7,39,0.532296,0.531021,0.075182,0.075182,5,0.882172
257,combined_raw_pca_sparsepca,50,17,36,0.524485,0.533980,0.000000,0.000000,6,0.882172
256,combined_raw_pca_sparsepca,50,16,29,0.541093,0.526178,0.000000,0.000000,6,0.882172
255,combined_raw_pca_sparsepca,50,15,37,0.542017,0.539712,0.000000,0.000000,6,0.882172
254,combined_raw_pca_sparsepca,50,14,12,0.533480,0.539348,0.000000,0.000000,6,0.882172
253,combined_raw_pca_sparsepca,50,13,36,0.533507,0.539314,0.000000,0.000000,6,0.882172


Raw axis vs dense probe vs best task-guided discovery


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,discovery_auc,discovered_direction_idx
0,18,0.687900,0.705692,1,0.700613,0.687900,42
1,2,0.725691,0.648612,2,0.738841,0.725691,51
2,9,0.586024,0.230210,3,0.628799,0.586024,16
3,19,0.527459,0.150507,4,0.575840,0.532505,39
4,7,0.522412,0.075182,5,0.571976,0.532296,39
17,17,0.532613,0.000000,6,0.580520,0.524485,36
16,16,0.515068,0.000000,6,0.577804,0.541093,29
15,15,0.516571,0.000000,6,0.572303,0.542017,37
14,14,0.533480,0.000000,6,0.569513,0.533480,12
13,13,0.530722,0.000000,6,0.586204,0.533507,36


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.687900,0.666538,0.609897
1,dense_probe,0.700613,0.689417,0.643214
2,task_guided_discovery,0.687900,0.666538,0.612883


## Strict Top-k Residual Probe Recovery

This supervised diagnostic asks whether each hidden concept is recoverable from only the top-k residual dimensions selected by a dense probe. It separates axis-like, sparse-mixed, and broadly distributed residual encodings.


In [813]:
# -----------------------------
# Strict top-k residual probe recovery
# -----------------------------
# Dense probes show whether hidden concept information is linearly present in the
# residual channel. This section asks a stricter question: how many residual
# dimensions are needed to recover each hidden concept?
#
# Protocol for each hidden concept h:
# 1. Fit a dense logistic probe h ~ res_mu on train.
# 2. Rank residual dimensions by absolute dense-probe coefficient.
# 3. Refit probes using only the top-k ranked dimensions.
# 4. Evaluate every fixed top-k subset on test.
#
# Hidden labels are used here, so this is a recoverability diagnostic rather
# than an unsupervised discovery method.

TOPK_PROBE_KS = [1, 2, 3, 5, 10, 20, 50]


def fit_topk_probe_for_hidden(X_train, y_train, X_test, y_test, ks=TOPK_PROBE_KS):
    """Rank residual dimensions with a dense probe, then test strict top-k refits.
    Only fit with the top-k dimensions with highest absolute coefficient in the dense probe.
    """
    n_dims = X_train.shape[1]
    ks = sorted({min(k, n_dims) for k in ks})

    dense_clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
    )
    dense_clf.fit(X_train, y_train)
    dense_prob = dense_clf.predict_proba(X_test)[:, 1]
    dense_auc = roc_auc_score(y_test, dense_prob)

    coef = dense_clf.named_steps["logisticregression"].coef_[0]
    
    ranked_dims = np.argsort(-np.abs(coef)).astype(int).tolist()

    rows = []
    for k in ks:
        # Select the top-k dimensions based on the dense probe's absolute coefficient ranking.

        selected_dims = ranked_dims[:k]
        topk_clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
        )
        topk_clf.fit(X_train[:, selected_dims], y_train)

        prob = topk_clf.predict_proba(X_test[:, selected_dims])[:, 1]
        pred = (prob >= 0.5).astype(int)
        rows.append({
            "k": k,
            "topk_auc": roc_auc_score(y_test, prob),
            "topk_accuracy": accuracy_score(y_test, pred),
            "topk_f1": f1_score(y_test, pred, zero_division=0),
            "dense_auc_from_same_probe": dense_auc,
            "topk_gap_to_dense": dense_auc - roc_auc_score(y_test, prob),
            "selected_dims": selected_dims,
        })

    return rows, ranked_dims, coef


def strict_topk_probe_train_test(train_scores, train_hidden, test_scores, test_hidden, ks=TOPK_PROBE_KS):
    X_train = to_numpy(train_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)
    H_train = to_numpy(train_hidden).astype(int)
    H_test = to_numpy(test_hidden).astype(int)

    rows = []
    coef_rows = []
    for h in range(H_train.shape[1]):
        if len(np.unique(H_train[:, h])) < 2 or len(np.unique(H_test[:, h])) < 2:
            continue

        topk_rows, ranked_dims, coef = fit_topk_probe_for_hidden(
            X_train,
            H_train[:, h],
            X_test,
            H_test[:, h],
            ks=ks,
        )
        for row in topk_rows:
            rows.append({"hidden_idx": h, **row})

        coef_rows.append({
            "hidden_idx": h,
            "ranked_dims_by_dense_coef": ranked_dims,
            "top10_dense_coef_dims": ranked_dims[:10],
            "dense_coef_l1_norm": float(np.abs(coef).sum()),
            "dense_coef_l2_norm": float(np.sqrt((coef ** 2).sum())),
        })

    return pd.DataFrame(rows), pd.DataFrame(coef_rows)


topk_probe_long, topk_probe_coef_info = strict_topk_probe_train_test(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)

topk_probe_long = attach_relevance(topk_probe_long, relevance)

# Wide table: one row per hidden concept, one AUC column per k.
topk_auc_wide = (
    topk_probe_long
    .pivot(index="hidden_idx", columns="k", values="topk_auc")
    .rename(columns=lambda k: f"top{k}_auc")
    .reset_index()
)

topk_gap_wide = (
    topk_probe_long
    .pivot(index="hidden_idx", columns="k", values="topk_gap_to_dense")
    .rename(columns=lambda k: f"top{k}_gap_to_dense")
    .reset_index()
)

topk_probe_recovery_eval = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(topk_auc_wide, on="hidden_idx", how="left")
    .merge(topk_gap_wide, on="hidden_idx", how="left")
    .merge(topk_probe_coef_info, on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

candidate_display_cols = [
    "hidden_idx",
    "abs_w_hid",
    "rank_abs_w",
    "axis_auc",
    "top1_auc",
    "top2_auc",
    "top3_auc",
    "top5_auc",
    "top10_auc",
    "top20_auc",
    "top50_auc",
    "distributed_auc",
    "top10_dense_coef_dims",
]
display_cols = [col for col in candidate_display_cols if col in topk_probe_recovery_eval.columns]

print("Strict top-k residual probe recovery ranked by abs(w_hid)")
display(topk_probe_recovery_eval[display_cols])

# Summarize top-k hidden concepts by true task weight, for each strict residual subset size.
summary_rows = []
for hidden_k in [1, 3, 5, 10]:
    top_hidden = topk_probe_recovery_eval.head(hidden_k)
    row = {"top_hidden_by_abs_w": hidden_k}
    row["raw_axis"] = top_hidden["axis_auc"].mean()
    for residual_k in TOPK_PROBE_KS:
        col = f"top{min(residual_k, to_numpy(splits['train']['res_mu']).shape[1])}_auc"
        row[f"top{residual_k}_residual_dims"] = top_hidden[col].mean()
    row["dense_probe"] = top_hidden["distributed_auc"].mean()
    summary_rows.append(row)

topk_probe_summary = pd.DataFrame(summary_rows)
print("Mean hidden recovery as residual subset size increases")
display(topk_probe_summary)

# Estimate how many residual dimensions are needed to get close to the dense probe.
# A hidden concept is considered close once top-k AUC is within 0.01/0.03/0.05 of dense.
close_rows = []
for _, row in topk_probe_recovery_eval.iterrows():
    out = {
        "hidden_idx": int(row["hidden_idx"]),
        "abs_w_hid": row["abs_w_hid"],
        "rank_abs_w": int(row["rank_abs_w"]),
        "axis_auc": row["axis_auc"],
        "dense_probe_auc": row["distributed_auc"],
    }
    for tol in [0.01, 0.03, 0.05]:
        needed = np.nan
        for residual_k in TOPK_PROBE_KS:
            col = f"top{min(residual_k, to_numpy(splits['train']['res_mu']).shape[1])}_auc"
            if row[col] >= row["distributed_auc"] - tol:
                needed = residual_k
                break
        out[f"dims_needed_within_{tol:.2f}_auc"] = needed
    close_rows.append(out)

topk_compactness = pd.DataFrame(close_rows).sort_values("rank_abs_w")
print("Residual dimensions needed to approach dense-probe AUC")
display(topk_compactness)


Strict top-k residual probe recovery ranked by abs(w_hid)


,hidden_idx,abs_w_hid,rank_abs_w,axis_auc,top1_auc,top2_auc,top3_auc,top5_auc,top10_auc,top20_auc,distributed_auc,top10_dense_coef_dims
0,18,0.705692,1,0.687900,0.678221,0.685898,0.688247,0.696596,0.700079,0.700613,0.700613,"[0, 16, 17, 1, 19, 6, 11, 9, 10, 2]"
1,2,0.648612,2,0.725691,0.716406,0.718380,0.725212,0.725192,0.737593,0.738841,0.738841,"[1, 14, 15, 19, 11, 16, 2, 7, 6, 18]"
2,9,0.230210,3,0.586024,0.586024,0.602535,0.606095,0.607762,0.627045,0.628799,0.628799,"[16, 1, 0, 17, 8, 11, 15, 6, 7, 12]"
3,19,0.150507,4,0.527459,0.516273,0.547077,0.557583,0.560390,0.573911,0.575840,0.575840,"[18, 16, 14, 5, 3, 7, 1, 12, 2, 10]"
4,7,0.075182,5,0.522412,0.517701,0.516871,0.538684,0.556922,0.565395,0.571976,0.571976,"[14, 16, 4, 18, 5, 17, 13, 10, 7, 6]"
17,17,0.000000,6,0.532613,0.520661,0.555184,0.561982,0.568010,0.574991,0.580520,0.580520,"[1, 4, 9, 6, 19, 7, 17, 13, 5, 10]"
16,16,0.000000,6,0.515068,0.506364,0.544779,0.545061,0.557058,0.575065,0.577804,0.577804,"[13, 2, 8, 1, 17, 0, 10, 16, 4, 19]"
15,15,0.000000,6,0.516571,0.517068,0.530211,0.535311,0.551975,0.568949,0.572303,0.572303,"[13, 14, 8, 1, 12, 3, 2, 6, 10, 18]"
14,14,0.000000,6,0.533480,0.535524,0.547836,0.552796,0.560427,0.565127,0.569513,0.569513,"[2, 9, 1, 14, 6, 0, 16, 3, 5, 8]"
13,13,0.000000,6,0.530722,0.518285,0.519196,0.547931,0.567657,0.580815,0.586204,0.586204,"[2, 4, 13, 18, 10, 14, 5, 3, 0, 15]"


Mean hidden recovery as residual subset size increases


,top_hidden_by_abs_w,raw_axis,top1_residual_dims,top2_residual_dims,top3_residual_dims,top5_residual_dims,top10_residual_dims,top20_residual_dims,top50_residual_dims,dense_probe
0,1,0.687900,0.678221,0.685898,0.688247,0.696596,0.700079,0.700613,0.700613,0.700613
1,3,0.666538,0.660217,0.668938,0.673185,0.676517,0.688239,0.689417,0.689417,0.689417
2,5,0.609897,0.602925,0.614152,0.623164,0.629372,0.640805,0.643214,0.643214,0.643214
3,10,0.567794,0.561253,0.576797,0.585890,0.595199,0.606897,0.610241,0.610241,0.610241


Residual dimensions needed to approach dense-probe AUC


,hidden_idx,abs_w_hid,rank_abs_w,axis_auc,dense_probe_auc,dims_needed_within_0.01_auc,dims_needed_within_0.03_auc,dims_needed_within_0.05_auc
0,18,0.705692,1,0.687900,0.700613,5,1,1
1,2,0.648612,2,0.725691,0.738841,10,1,1
2,9,0.230210,3,0.586024,0.628799,10,2,1
3,19,0.150507,4,0.527459,0.575840,10,2,2
4,7,0.075182,5,0.522412,0.571976,10,5,3
17,4,0.000000,6,0.504254,0.578416,10,2,2
16,5,0.000000,6,0.502981,0.596214,10,10,3
15,6,0.000000,6,0.503160,0.595253,10,10,5
14,8,0.000000,6,0.534018,0.579181,10,3,3
13,3,0.000000,6,0.519115,0.582369,10,2,2


## SAE Residual Concept Discovery

This section adapts the BatchTopK SAE idea from the CEM concept-discovery repo to the Residual SCBM residual channel. The SAE is trained unsupervised on `train/res_mu`; validation chooses SAE-feature/hidden-concept matches; test evaluates those fixed matches.


In [814]:
# -----------------------------
# SAE configuration
# -----------------------------
# The SAE is trained only on residual-channel representations (`res_mu`).
# It does not see hidden concept labels during training.
SAE_CONFIG = {
    # The SAE dictionary is overcomplete: for an R-dimensional residual channel,
    # the SAE has dict_size_multiplier * R candidate sparse features.
    # For R=50 and multiplier=4, this gives 200 possible discovered features.
    "dict_size_multiplier": 4,

    # BatchTopK sparsity: across each batch, keep top_k active features per
    # example on average. Smaller values force more sparse/competitive features.
    "top_k": 4,

    # Optimization hyperparameters.
    "batch_size": 2048,
    "n_epochs": 50,
    "lr": 1e-3,
    "beta1": 0.9,
    "beta2": 0.999,
    "l1_coeff": 0.0,
    "max_grad_norm": 1.0,
    "seed": 0,

    # After training, discard SAE features that almost never activate on train.
    "min_train_active_rate": 1e-4,
}


class ResidualBatchTopKSAE(nn.Module):
    """
    BatchTopK sparse autoencoder for residual-channel concept discovery.

    This mirrors the SAE idea used in the CEM concept-discovery repo: learn an
    overcomplete dictionary that reconstructs the representation, while forcing
    only a small number of features to activate. Nonzero SAE activations are then
    treated as candidate discovered residual concepts.
    """

    def __init__(self, input_dim, dict_size, top_k):
        super().__init__()
        self.input_dim = input_dim
        self.dict_size = dict_size
        self.top_k = top_k

        # Decoder bias is the reconstruction baseline. Encoder bias shifts SAE
        # feature thresholds. W_enc maps residual vectors to feature activations;
        # W_dec maps sparse feature activations back to residual space.
        self.b_dec = nn.Parameter(torch.zeros(input_dim))
        self.b_enc = nn.Parameter(torch.zeros(dict_size))
        self.W_enc = nn.Parameter(torch.empty(input_dim, dict_size))
        self.W_dec = nn.Parameter(torch.empty(dict_size, input_dim))

        # Initialize decoder as tied to encoder transpose, then normalize decoder
        # atoms so feature scale is controlled by activations rather than weights.
        nn.init.kaiming_uniform_(self.W_enc)
        self.W_dec.data[:] = self.W_enc.t().data
        self.renorm_decoder_weights()

    @torch.no_grad()
    def renorm_decoder_weights(self):
        # Keep each decoder feature vector at unit norm after optimizer updates.
        self.W_dec.data = self.W_dec.data / (self.W_dec.data.norm(dim=-1, keepdim=True) + 1e-8)

    def encode_dense(self, x):
        # Dense nonnegative feature activations before sparsification.
        return F.relu((x - self.b_dec) @ self.W_enc + self.b_enc)

    def sparsify(self, acts):
        # BatchTopK sparsification. Instead of keeping top_k per row, this keeps
        # top_k * batch_size activations over the entire batch. This creates
        # competition across both features and examples.
        k_total = min(self.top_k * acts.shape[0], acts.numel())
        top = torch.topk(acts.flatten(), k_total, dim=-1)
        return torch.zeros_like(acts.flatten()).scatter(-1, top.indices, top.values).reshape_as(acts)

    def forward(self, x):
        # Encode -> sparsify -> reconstruct. Training minimizes reconstruction
        # error only; no hidden concept labels are used here.
        acts_dense = self.encode_dense(x)
        acts_sparse = self.sparsify(acts_dense)
        x_hat = acts_sparse @ self.W_dec + self.b_dec
        return x_hat, acts_sparse


def standardize_from_train(train_scores, *other_scores):
    # Fit normalization on train residuals only, then apply the same transform to
    # val/test. This avoids leaking val/test distribution information into SAE training.
    train = torch.as_tensor(to_numpy(train_scores), dtype=torch.float32)
    mean = train.mean(dim=0, keepdim=True)
    std = train.std(dim=0, keepdim=True).clamp_min(1e-6)
    standardized = [(train - mean) / std]
    for scores in other_scores:
        x = torch.as_tensor(to_numpy(scores), dtype=torch.float32)
        standardized.append((x - mean) / std)
    return standardized, mean, std


def train_residual_sae(train_scores, cfg):
    # Unsupervised SAE training on train residual representations.
    torch.manual_seed(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X = torch.as_tensor(to_numpy(train_scores), dtype=torch.float32)
    input_dim = X.shape[1]
    dict_size = int(cfg["dict_size_multiplier"] * input_dim)

    model = ResidualBatchTopKSAE(input_dim=input_dim, dict_size=dict_size, top_k=cfg["top_k"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], betas=(cfg["beta1"], cfg["beta2"]))

    history = []
    n = X.shape[0]
    batch_size = cfg["batch_size"]

    for epoch in range(cfg["n_epochs"]):
        # Shuffle train residuals each epoch. This is still unsupervised: only X is used.
        perm = torch.randperm(n)
        epoch_loss = 0.0
        # Track mean L0 "activation" (number of nonzero features) as a diagnostic, even though sparsity is really controlled by BatchTopK.
        epoch_l0 = 0.0
        seen = 0
        model.train()

        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            xb = X[idx].to(device)
            # x_hat is the reconstructed residual, acts are the sparse SAE features. Both have shape batch_size x dict_size.
            x_hat, acts = model(xb)

            # Reconstruction loss encourages SAE features to preserve residual information.
            # l1_coeff is currently zero because BatchTopK already enforces sparsity.
            l2 = (x_hat - xb).pow(2).mean()
            l1 = cfg["l1_coeff"] * acts.abs().sum(dim=1).mean()
            loss = l2 + l1

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
            opt.step()
            model.renorm_decoder_weights()

            bsz = xb.shape[0]
            epoch_loss += loss.item() * bsz
            epoch_l0 += (acts > 0).float().sum(dim=1).mean().item() * bsz
            seen += bsz

        history.append({
            "epoch": epoch + 1,
            "loss": epoch_loss / seen,
            "mean_l0": epoch_l0 / seen,
        })

        if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == cfg["n_epochs"]:
            print(f"epoch {epoch + 1:03d} | loss {history[-1]['loss']:.5f} | mean_l0 {history[-1]['mean_l0']:.2f}")

    return model, pd.DataFrame(history)


@torch.no_grad()
def sae_feature_activations(model, scores, batch_size=4096):
    # Run trained SAE on a split and return sparse feature activations.
    # Shape: n_examples x n_sae_features.
    device = next(model.parameters()).device
    X = torch.as_tensor(to_numpy(scores), dtype=torch.float32)
    acts = []
    model.eval()
    for start in range(0, X.shape[0], batch_size):
        xb = X[start:start + batch_size].to(device)
        _, batch_acts = model(xb)
        acts.append(batch_acts.cpu())
    return torch.cat(acts, dim=0).numpy()


def filter_live_sae_features(train_acts, val_acts, test_acts, min_active_rate=1e-4):
    # Drop dead/near-dead features based on train activations only, then apply the
    # same feature mask to val/test.
    train_active_rate = (train_acts > 0).mean(axis=0)
    live_mask = train_active_rate >= min_active_rate
    live_feature_ids = np.flatnonzero(live_mask)
    return (
        train_acts[:, live_mask],
        val_acts[:, live_mask],
        test_acts[:, live_mask],
        live_feature_ids,
        train_active_rate[live_mask],
    )


def best_binary_threshold(y_true, score):
    # Choose a validation threshold for accuracy/F1. AUC itself is threshold-free,
    # but accuracy/F1 need a cutoff.
    y_true = to_numpy(y_true).astype(int)
    score = to_numpy(score).astype(float)
    candidates = np.unique(np.quantile(score, np.linspace(0.01, 0.99, 99)))
    if len(candidates) == 0:
        return 0.0, np.nan, np.nan

    best_thr = float(candidates[0])
    best_acc = -np.inf
    best_f1 = np.nan
    for thr in candidates:
        pred = (score >= thr).astype(int)
        acc = accuracy_score(y_true, pred)
        if acc > best_acc:
            best_thr = float(thr)
            best_acc = float(acc)
            best_f1 = float(f1_score(y_true, pred, zero_division=0))
    return best_thr, best_acc, best_f1


def match_hidden_to_sae_features(feature_acts, hidden_residuals):
    # Validation-time matching: for each true hidden concept, find the SAE feature
    # whose activation best separates h=1 from h=0 by orientation-free AUC.
    # This uses hidden labels only for evaluation/model selection, not SAE training.
    auc, directions = residual_hidden_auc_matrix(feature_acts, hidden_residuals)
    A = to_numpy(feature_acts).astype(float)
    H = to_numpy(hidden_residuals).astype(int)
    rows = []

    for h in range(auc.shape[1]):
        f = int(np.nanargmax(auc[:, h]))
        score = A[:, f]
        if directions[f, h] == "negative":
            score = -score
        threshold, val_accuracy, val_f1 = best_binary_threshold(H[:, h], score)
        rows.append({
            "hidden_idx": h,
            "sae_feature_idx": f,
            "auc": float(auc[f, h]),
            "direction": directions[f, h],
            "threshold": threshold,
            "val_accuracy_at_threshold": val_accuracy,
            "val_f1_at_threshold": val_f1,
        })
    return pd.DataFrame(rows), auc, directions


def evaluate_fixed_sae_matches(feature_acts, hidden_residuals, matches):
    # Test-time evaluation: use the validation-chosen SAE feature, orientation,
    # and threshold, then evaluate on held-out test hidden labels.
    A = to_numpy(feature_acts).astype(float)
    H = to_numpy(hidden_residuals).astype(int)
    rows = []

    for row in matches.itertuples(index=False):
        h = int(row.hidden_idx)
        f = int(row.sae_feature_idx)
        score = A[:, f]
        if row.direction == "negative":
            score = -score

        pred = (score >= float(row.threshold)).astype(int)
        rows.append({
            "hidden_idx": h,
            "sae_feature_idx": f,
            "direction": row.direction,
            "threshold": float(row.threshold),
            "sae_auc": roc_auc_score(H[:, h], score),
            "sae_accuracy_at_threshold": accuracy_score(H[:, h], pred),
            "sae_f1_at_threshold": f1_score(H[:, h], pred, zero_division=0),
        })

    return pd.DataFrame(rows)


In [815]:
# -----------------------------
# Train one SAE and compute activations
# -----------------------------
# Standardize residual means using train statistics only. The SAE sees only
# train residual vectors during training; val/test are transformed later for evaluation.
(sae_inputs, sae_train_mean, sae_train_std) = standardize_from_train(
    splits["train"]["res_mu"],
    splits["val"]["res_mu"],
    splits["test"]["res_mu"],
)
X_train_sae, X_val_sae, X_test_sae = sae_inputs

# Train the SAE unsupervised on train residual channel means.
sae_model, sae_history = train_residual_sae(X_train_sae, SAE_CONFIG)
display(sae_history.tail())

# Apply the trained SAE to each split. These activations are the candidate
# discovered residual concepts. Hidden labels are still not used here.
train_sae_acts = sae_feature_activations(sae_model, X_train_sae, SAE_CONFIG["batch_size"])
val_sae_acts = sae_feature_activations(sae_model, X_val_sae, SAE_CONFIG["batch_size"])
test_sae_acts = sae_feature_activations(sae_model, X_test_sae, SAE_CONFIG["batch_size"])

# Remove dead features based on train activity. Keep track of the original SAE
# dictionary ids so evaluation tables can report interpretable feature ids.
(
    train_sae_acts_live,
    val_sae_acts_live,
    test_sae_acts_live,
    live_sae_feature_ids,
    live_sae_active_rate,
) = filter_live_sae_features(
    train_sae_acts,
    val_sae_acts,
    test_sae_acts,
    min_active_rate=SAE_CONFIG["min_train_active_rate"],
)

print(f"SAE dictionary size: {train_sae_acts.shape[1]}")
print(f"Live SAE features: {len(live_sae_feature_ids)}")
print(f"Mean live feature active rate: {live_sae_active_rate.mean():.4f}")


epoch 001 | loss 0.68794 | mean_l0 4.00
epoch 010 | loss 0.03931 | mean_l0 4.00
epoch 020 | loss 0.02371 | mean_l0 4.00
epoch 030 | loss 0.01929 | mean_l0 4.00
epoch 040 | loss 0.01649 | mean_l0 4.00
epoch 050 | loss 0.01550 | mean_l0 4.00


,epoch,loss,mean_l0
45,46,0.015870,4.0
46,47,0.015782,4.0
47,48,0.015687,4.0
48,49,0.015603,4.0
49,50,0.015500,4.0


SAE dictionary size: 80
Live SAE features: 73
Mean live feature active rate: 0.0548


## SAE Hyperparameter Tuning

The first SAE run can collapse several hidden concepts onto one broad SAE feature. This grid searches over dictionary size, sparsity `top_k`, epochs, and seed, then ranks configs by recovery of the highest `abs(w_hid)` hidden concepts.


In [816]:
# -----------------------------
# SAE hyperparameter tuning
# -----------------------------
# Starter grid. Expand this once the mechanics look sensible.
# Full suggested grid:
#   dict_size_multiplier: [4, 8, 12]
#   top_k: [1, 2, 4, 8]
#   seed: [0, 1, 2]
SAE_TUNING_GRID = [
    {"dict_size_multiplier": d, "top_k": k, "seed": seed}
    for d in [4, 8]
    for k in [1, 2, 4]
    for seed in [0, 1]
]

SAE_TUNING_BASE_CONFIG = {
    **SAE_CONFIG,
    "n_epochs": 75,
}


def ensure_sae_inputs_available():
    """Create standardized SAE inputs if the single-SAE cell has not been run."""
    global X_train_sae, X_val_sae, X_test_sae, sae_train_mean, sae_train_std

    if all(name in globals() for name in ["X_train_sae", "X_val_sae", "X_test_sae"]):
        return X_train_sae, X_val_sae, X_test_sae

    (sae_inputs, sae_train_mean, sae_train_std) = standardize_from_train(
        splits["train"]["res_mu"],
        splits["val"]["res_mu"],
        splits["test"]["res_mu"],
    )
    X_train_sae, X_val_sae, X_test_sae = sae_inputs
    return X_train_sae, X_val_sae, X_test_sae


def evaluate_sae_config(cfg):
    """
    Train one SAE config and evaluate discovered SAE features.

    The SAE is trained without hidden labels. Hidden labels are used only to
    choose feature/hidden matches on validation and evaluate fixed matches on test.
    """
    X_train, X_val, X_test = ensure_sae_inputs_available()

    model, history = train_residual_sae(X_train, cfg)

    train_acts = sae_feature_activations(model, X_train, cfg["batch_size"])
    val_acts = sae_feature_activations(model, X_val, cfg["batch_size"])
    test_acts = sae_feature_activations(model, X_test, cfg["batch_size"])

    # Filter to "live" SAE features that are active on at least a minimum fraction of train examples.
    # Remove dead SAE features that are never active
    train_live, val_live, test_live, live_ids, live_active_rate = filter_live_sae_features(
        train_acts,
        val_acts,
        test_acts,
        min_active_rate=cfg["min_train_active_rate"],
    )

    matches_val, _, _ = match_hidden_to_sae_features(
        val_live,
        splits["val"]["hidden_residuals"],
    )
    test_eval_cfg = evaluate_fixed_sae_matches(
        test_live,
        splits["test"]["hidden_residuals"],
        matches_val,
    )
    test_eval_cfg["sae_dict_feature_idx"] = live_ids[test_eval_cfg["sae_feature_idx"].values]
    test_eval_cfg = attach_relevance(test_eval_cfg, relevance).sort_values("rank_abs_w")

    top1 = test_eval_cfg.head(1)
    top3 = test_eval_cfg.head(3)
    top5 = test_eval_cfg.head(5)
    top10 = test_eval_cfg.head(10)

    matched_top5 = top5["sae_dict_feature_idx"].tolist()
    unique_top5 = len(set(matched_top5))

    summary = {
        "dict_size_multiplier": cfg["dict_size_multiplier"],
        "dict_size": int(cfg["dict_size_multiplier"] * to_numpy(splits["train"]["res_mu"]).shape[1]),
        "top_k": cfg["top_k"],
        "seed": cfg["seed"],
        "n_epochs": cfg["n_epochs"],
        "final_loss": float(history["loss"].iloc[-1]),
        "live_features": int(len(live_ids)),
        "mean_live_active_rate": float(live_active_rate.mean()) if len(live_active_rate) else np.nan,
        "top1_mean_sae_auc": float(top1["sae_auc"].mean()),
        "top3_mean_sae_auc": float(top3["sae_auc"].mean()),
        "top5_mean_sae_auc": float(top5["sae_auc"].mean()),
        "top10_mean_sae_auc": float(top10["sae_auc"].mean()),
        "top5_unique_matched_features": unique_top5,
        "top5_feature_ids": matched_top5,
        "spearman_abs_w_sae_auc": float(test_eval_cfg[["abs_w_hid", "sae_auc"]].corr(method="spearman").iloc[0, 1]),
    }

    return summary, test_eval_cfg, history


# Store one summary row per config, plus detailed per-hidden-concept tables.
sae_tuning_summaries = []
sae_tuning_details = {}
sae_tuning_histories = {}

for i, overrides in enumerate(SAE_TUNING_GRID, start=1):
    cfg = {**SAE_TUNING_BASE_CONFIG, **overrides}
    key = f"d{cfg['dict_size_multiplier']}_k{cfg['top_k']}_seed{cfg['seed']}_ep{cfg['n_epochs']}"
    print("=" * 80)
    print(f"[{i}/{len(SAE_TUNING_GRID)}] {key}")

    # Train/evaluate this config. Hidden labels are used only inside validation/test matching.
    summary, detail, history = evaluate_sae_config(cfg)
    sae_tuning_summaries.append(summary)
    sae_tuning_details[key] = detail
    sae_tuning_histories[key] = history

# Rank configs first by top-5 recovery of the highest-weight hidden concepts,
# then prefer less feature collapse when AUC is tied.
sae_tuning_results = pd.DataFrame(sae_tuning_summaries).sort_values(
    ["top5_mean_sae_auc", "top5_unique_matched_features", "top3_mean_sae_auc"],
    ascending=[False, False, False],
)

print("SAE tuning results ranked by top-5 task-relevant recovery")
display(sae_tuning_results)

# Inspect the best config in detail and compare it to raw-axis/probe baselines.
best_row = sae_tuning_results.iloc[0]
best_sae_key = f"d{int(best_row['dict_size_multiplier'])}_k{int(best_row['top_k'])}_seed{int(best_row['seed'])}_ep{int(best_row['n_epochs'])}"
print("Best SAE config key:", best_sae_key)
print("Best SAE config detail ranked by abs(w_hid)")
display(sae_tuning_details[best_sae_key][[
    "hidden_idx",
    "sae_dict_feature_idx",
    "sae_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
    "sae_accuracy_at_threshold",
    "sae_f1_at_threshold",
]])

best_comparison = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(sae_tuning_details[best_sae_key][["hidden_idx", "sae_auc", "sae_dict_feature_idx"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

print("Best tuned SAE vs raw axis vs distributed probe")
display(best_comparison)

tuned_comparison_summary = pd.DataFrame({
    "method": ["raw_axis", "distributed_probe", "best_tuned_sae"],
    "top1_mean_auc": [
        best_comparison.head(1)["axis_auc"].mean(),
        best_comparison.head(1)["distributed_auc"].mean(),
        best_comparison.head(1)["sae_auc"].mean(),
    ],
    "top3_mean_auc": [
        best_comparison.head(3)["axis_auc"].mean(),
        best_comparison.head(3)["distributed_auc"].mean(),
        best_comparison.head(3)["sae_auc"].mean(),
    ],
    "top5_mean_auc": [
        best_comparison.head(5)["axis_auc"].mean(),
        best_comparison.head(5)["distributed_auc"].mean(),
        best_comparison.head(5)["sae_auc"].mean(),
    ],
})
display(tuned_comparison_summary)


[1/12] d4_k1_seed0_ep75
epoch 001 | loss 0.69506 | mean_l0 1.00
epoch 010 | loss 0.18747 | mean_l0 1.00
epoch 020 | loss 0.07366 | mean_l0 1.00
epoch 030 | loss 0.03186 | mean_l0 1.00
epoch 040 | loss 0.02516 | mean_l0 1.00
epoch 050 | loss 0.02379 | mean_l0 1.00
epoch 060 | loss 0.02349 | mean_l0 1.00
epoch 070 | loss 0.02335 | mean_l0 1.00
epoch 075 | loss 0.02329 | mean_l0 1.00
[2/12] d4_k1_seed1_ep75
epoch 001 | loss 0.60398 | mean_l0 1.00
epoch 010 | loss 0.14025 | mean_l0 1.00
epoch 020 | loss 0.07549 | mean_l0 1.00
epoch 030 | loss 0.04397 | mean_l0 1.00
epoch 040 | loss 0.03184 | mean_l0 1.00
epoch 050 | loss 0.02762 | mean_l0 1.00
epoch 060 | loss 0.02416 | mean_l0 1.00
epoch 070 | loss 0.02349 | mean_l0 1.00
epoch 075 | loss 0.02334 | mean_l0 1.00
[3/12] d4_k2_seed0_ep75
epoch 001 | loss 0.66390 | mean_l0 2.00
epoch 010 | loss 0.07099 | mean_l0 2.00
epoch 020 | loss 0.04085 | mean_l0 2.00
epoch 030 | loss 0.02910 | mean_l0 2.00
epoch 040 | loss 0.02458 | mean_l0 2.00
epoch 05

,dict_size_multiplier,dict_size,top_k,seed,n_epochs,final_loss,live_features,mean_live_active_rate,top1_mean_sae_auc,top3_mean_sae_auc,top5_mean_sae_auc,top10_mean_sae_auc,top5_unique_matched_features,top5_feature_ids,spearman_abs_w_sae_auc
4,4,80,4,0,75,0.014278,72,0.055552,0.675150,0.647239,0.597265,0.554303,4,"[39, 39, 34, 12, 0]",0.628472
5,4,80,4,1,75,0.012827,73,0.054793,0.670880,0.645309,0.596147,0.553617,3,"[67, 18, 67, 19, 18]",0.626496
11,8,160,4,1,75,0.014294,81,0.049369,0.672161,0.645168,0.595856,0.553543,5,"[57, 20, 85, 103, 157]",0.628472
3,4,80,2,1,75,0.023410,32,0.062470,0.672300,0.645290,0.595655,0.551861,3,"[67, 67, 67, 19, 25]",0.628472
10,8,160,4,0,75,0.013197,103,0.038826,0.671188,0.643754,0.595303,0.553263,3,"[56, 64, 72, 72, 64]",0.610685
8,8,160,2,0,75,0.020120,31,0.064472,0.672989,0.645162,0.595225,0.552350,3,"[56, 64, 56, 56, 92]",0.628472
2,4,80,2,0,75,0.020183,49,0.040807,0.670548,0.643835,0.595062,0.553561,4,"[39, 64, 39, 12, 73]",0.628472
9,8,160,2,1,75,0.021307,32,0.062459,0.660523,0.643154,0.594801,0.552196,3,"[157, 157, 57, 103, 157]",0.644283
1,4,80,1,1,75,0.023341,10,0.099923,0.668014,0.643009,0.594607,0.550523,2,"[67, 18, 67, 67, 18]",0.628472
6,8,160,1,0,75,0.023416,6,0.166639,0.665525,0.642147,0.594109,0.550284,2,"[56, 64, 56, 56, 64]",0.644283


Best SAE config key: d4_k4_seed0_ep75
Best SAE config detail ranked by abs(w_hid)


,hidden_idx,sae_dict_feature_idx,sae_auc,w_hid,abs_w_hid,rank_abs_w,sae_accuracy_at_threshold,sae_f1_at_threshold
18,18,39,0.675150,-0.705692,0.705692,1,0.6433,0.635983
2,2,39,0.692997,-0.648612,0.648612,2,0.6646,0.651641
9,9,34,0.573569,0.230210,0.230210,3,0.5535,0.487076
19,19,12,0.526891,-0.150507,0.150507,4,0.5151,0.647576
7,7,0,0.517717,0.075182,0.075182,5,0.5084,0.581759
4,4,76,0.503259,0.000000,0.000000,6,0.5075,0.093503
5,5,0,0.500138,0.000000,0.000000,6,0.5001,0.286774
6,6,6,0.510408,0.000000,0.000000,6,0.5079,0.106734
8,8,39,0.530180,0.000000,0.000000,6,0.5222,0.578213
1,1,34,0.512726,0.000000,0.000000,6,0.5153,0.589342


Best tuned SAE vs raw axis vs distributed probe


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,sae_auc,sae_dict_feature_idx
0,18,0.687900,0.705692,1,0.700613,0.675150,39
1,2,0.725691,0.648612,2,0.738841,0.692997,39
2,9,0.586024,0.230210,3,0.628799,0.573569,34
3,19,0.527459,0.150507,4,0.575840,0.526891,12
4,7,0.522412,0.075182,5,0.571976,0.517717,0
17,17,0.532613,0.000000,6,0.580520,0.523836,12
16,16,0.515068,0.000000,6,0.577804,0.510255,39
15,15,0.516571,0.000000,6,0.572303,0.518065,12
14,14,0.533480,0.000000,6,0.569513,0.527572,34
13,13,0.530722,0.000000,6,0.586204,0.524742,39


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.687900,0.666538,0.609897
1,distributed_probe,0.700613,0.689417,0.643214
2,best_tuned_sae,0.675150,0.647239,0.597265


## Summary of Recovery Approaches

This cell collects the top-1/top-3/top-5 hidden-concept recovery AUCs for every approach that has been run in the notebook. Rows are added only when the corresponding result objects exist.


In [817]:
# -----------------------------
# Summary of all hidden-concept recovery approaches
# -----------------------------
# Each row summarizes recovery of the hidden concepts ranked by abs(w_hid).
# top1/top3/top5 mean AUC therefore answers: among the most task-relevant
# hidden concepts, how well did this method recover them?
#
# This cell is intentionally defensive: it only includes methods whose result
# tables already exist in the notebook state. You can rerun it after running more
# sections and the table will automatically expand.

summary_rows = []


def add_topk_auc_summary(method_name, df, score_col):
    """Append top-1/top-3/top-5 mean AUC for a method, if its table exists."""
    if df is None or score_col not in df.columns:
        return

    # Defensively keep one row per hidden concept. Some upstream sweeps can
    # produce repeated effective top_m values when requested top_m exceeds the
    # number of available directions; duplicates would otherwise make top-3
    # mean AUC count the same hidden concept multiple times.
    ordered = df.sort_values(["rank_abs_w", score_col], ascending=[True, False]).drop_duplicates("hidden_idx")
    summary_rows.append({
        "method": method_name,
        "top1_mean_auc": ordered.head(1)[score_col].mean(),
        "top3_mean_auc": ordered.head(3)[score_col].mean(),
        "top5_mean_auc": ordered.head(5)[score_col].mean(),
    })


# Raw residual axis baseline: validation chooses one residual axis per hidden concept.
if "recovery_eval" in globals():
    add_topk_auc_summary("raw_axis", recovery_eval, "axis_auc")

# Dense supervised hidden probe: upper-bound style diagnostic for linear recoverability.
if "probe_recovery_eval" in globals():
    add_topk_auc_summary("distributed_probe", probe_recovery_eval, "distributed_auc")

# Strict top-k residual probes: supervised diagnostic of how many residual dimensions are needed.
if "topk_probe_recovery_eval" in globals():
    for k in [1, 2, 3, 5, 10, 20, 50]:
        col = f"top{k}_auc"
        if col in topk_probe_recovery_eval.columns:
            add_topk_auc_summary(f"strict_top{k}_probe", topk_probe_recovery_eval, col)

if "pca_dictionary_recovery_eval" in globals():
    add_topk_auc_summary("pca_dictionary", pca_dictionary_recovery_eval, "pca_auc")

# Single SAE run.
if "sae_test_eval" in globals():
    add_topk_auc_summary("sae", sae_test_eval, "sae_auc")

# Best tuned SAE run.
if "best_sae_detail" in globals():
    add_topk_auc_summary("best_tuned_sae", best_sae_detail, "sae_auc")
elif "sae_tuning_results" in globals() and "sae_tuning_details" in globals():
    best_row = sae_tuning_results.iloc[0]
    best_key = f"d{int(best_row['dict_size_multiplier'])}_k{int(best_row['top_k'])}_seed{int(best_row['seed'])}_ep{int(best_row['n_epochs'])}"
    if best_key in sae_tuning_details:
        add_topk_auc_summary("best_tuned_sae", sae_tuning_details[best_key], "sae_auc")


# Task-guided residual direction discovery.
if "best_task_guided_detail" in globals():
    add_topk_auc_summary("task_guided_discovery", best_task_guided_detail, "discovery_auc")
elif "task_guided_summary" in globals() and "task_guided_recovery_eval" in globals():
    best_task_guided = task_guided_summary.iloc[0]
    detail = task_guided_recovery_eval[
        (task_guided_recovery_eval["generator"] == best_task_guided["generator"])
        & (task_guided_recovery_eval["top_m_task_selected_directions"] == best_task_guided["top_m_task_selected_directions"])
    ]
    add_topk_auc_summary("task_guided_discovery", detail, "discovery_auc")

approach_summary = pd.DataFrame(summary_rows)

# Keep a readable order when those methods are present.
method_order = [
    "raw_axis",
    "distributed_probe",
    "strict_top1_probe",
    "strict_top2_probe",
    "strict_top3_probe",
    "strict_top5_probe",
    "strict_top10_probe",
    "strict_top20_probe",
    "strict_top50_probe",
    "sae",
    "best_tuned_sae",
    "task_guided_discovery",
]
if not approach_summary.empty:
    approach_summary["_order"] = approach_summary["method"].map({m: i for i, m in enumerate(method_order)}).fillna(len(method_order))
    approach_summary = approach_summary.sort_values(["_order", "method"]).drop(columns="_order").reset_index(drop=True)

print("Summary of hidden-concept recovery approaches")
display(approach_summary)







Summary of hidden-concept recovery approaches


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.687900,0.666538,0.609897
1,distributed_probe,0.700613,0.689417,0.643214
2,strict_top1_probe,0.678221,0.660217,0.602925
3,strict_top2_probe,0.685898,0.668938,0.614152
4,strict_top3_probe,0.688247,0.673185,0.623164
5,strict_top5_probe,0.696596,0.676517,0.629372
6,strict_top10_probe,0.700079,0.688239,0.640805
7,strict_top20_probe,0.700613,0.689417,0.643214
8,sae,0.682315,0.676733,0.629773
9,best_tuned_sae,0.675150,0.647239,0.597265


## Interpretation Guide

- Strong validation-chosen test AUCs mean the residual channel itself contains axis-aligned discovered hidden concepts.
- If distributed probe AUC is much higher than axis-aligned AUC, hidden information is present but entangled across residual dimensions.
- If only task-relevant hidden residuals are recovered, that is expected: the residual channel is trained to preserve task-relevant missing information, not necessarily every hidden factor.
- The next step after this baseline is SAE discovery on `train/res_mu.pt`, with feature activations evaluated on val/test.


- Task relevance is ranked by `abs(w_hid)`, the absolute ground-truth hidden-task coefficient. This directly answers whether the residual channel recovers the hidden concepts the synthetic task intended to matter most.
- Signal/prevalence statistics are descriptive diagnostics for visibility or empirical variation; they are not used as the main task-importance metric.

- SAE improves over raw axis discovery if `sae_auc` is higher than `axis_auc` for the top-weight hidden concepts. The distributed probe remains an upper-bound style diagnostic: if probe AUC is high but SAE AUC is low, the information is present but the chosen SAE configuration did not find a useful sparse basis.

- SAE tuning should be judged by top-k `sae_auc` among the highest `abs(w_hid)` hidden concepts, plus feature-collapse diagnostics such as unique matched SAE features in the top-5.

- For hard datasets, hidden task-score recovery (`hidden_residual_signal @ w_hid`) is often more meaningful than binary hidden-concept recovery, because the generator uses continuous residual signal in the label score.

- L1 sparse probes diagnose whether hidden concepts are recoverable from a small supervised linear combination of residual dimensions. If sparse probes approach dense probes, the representation is sparse but rotated; if only dense probes work, it is distributed.
